In [ ]:
%config SqlMagic.autopolars = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

In [ ]:
%load_ext sql

# Dépendances


In [ ]:
import json
import math
import os
from datetime import datetime, timedelta
from itertools import product
from pathlib import Path
from zoneinfo import ZoneInfo
from dotenv import load_dotenv
import requests
import time

import branca.colormap as bcm
import duckdb
import folium
import geopandas as gpd
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import polars as pl
import polars_h3 as plh3
import polars_st as st
import shapely
from dotenv import load_dotenv
from folium import plugins
from sqlalchemy import create_engine

# Configuration


In [ ]:
USE_CACHED_ARTIFACTS = True
USE_CACHED_JOURNEYS_WITH_NEAREST_STATION = True


DEFAULT_START_DATE = datetime(2024, 9, 1, tzinfo=ZoneInfo("GMT"))
ENTITY_CONFIGS = {
    "driver": {
        "identity_col": "driver_identity_key",
        "first_trip_col": "first_trip_datetime",
        "label_plural": "conducteurs",
        "label_singular": "conducteur",
    },
    "passenger": {
        "identity_col": "passenger_identity_key",
        "first_trip_col": "passenger_first_trip_datetime", 
        "label_plural": "passagers",
        "label_singular": "passager",
    }
}

In [ ]:
load_dotenv()
DB_URL = os.environ.get("DB_URL")
IDF_GEOJSON = os.environ.get("IDF_GEOJSON")
BUDGET_IDFM = os.environ.get("BUDGET_IDFM")
DUREE_CAMPAGNE_IDFM = os.environ.get("DUREE_CAMPAGNE_IDFM")

In [ ]:
AOM_SIRET = "28750007800020"

In [ ]:
OUTPUT_PATH = Path("outputs_idfm")

## Labels

In [ ]:
labels_map = {
    "month": "Mois",
    "num_journeys": "Nombre de trajets",
    "share_journeys": "% des trajets",
    "num_journeys_incentived": "Nombre de trajets avec incitation",
    "num_journeys_with_incentive": "Nombre de trajets avec incitation",
    "num_journeys_intra_territory_incentived_trips": "Nombre de trajets incités intra",
    "num_journeys_inter_territory_incentived_trips": "Nombre de trajets incités inter",
    "operator": "Opérateur",
    "incentive_amount_avg": "Incitation moyenne",
    "driver_revenue_avg": "Revenu moyen conducteur",
    "passenger_contribution_avg": "Contribution moyenne passager",
    "incentive_amount_intra_avg": "Incitation moyenne intra",
    "driver_revenue_intra_avg": "Revenu moyen conducteur intra",
    "passenger_contribution_intra_avg": "Contribution moyenne passager intra",
    "incentive_amount_inter_avg": "Incitation moyenne inter",
    "driver_revenue_inter_avg": "Revenu moyen conducteur inter",
    "passenger_contribution_inter_avg": "Contribution moyenne passager inter",
    "incentive_amount_per_km_avg": "Montant moyen d'incitation par km",
    "passenger_contribution_per_km_avg": "Contribution moyenne passager par km",
    "driver_revenue_per_km_avg": "Revenu moyen conducteur par km",
    "day": "Jour",
    "week": "Semaine",
    "month": "Mois",
    "year_month": "Mois",
    "distance_avg": "Distance moyenne",
    "distance_km": "Distance [km]",
    "distance_incentived_trips_avg": "Distance moyenne [km] - trajets avec incentives",
    "campaign_type": "Campagne",
    "distance": "Distance",
    "num_journeys_with_aom_incentive": "Nombre de trajets incités par l'AOM",
    "num_journeys_with_operator_incentive": "Nombre de trajets incités par un opérateur",
    "num_journeys_intra_territory": "Nombre de trajets intra-territoire",
    "num_journeys_inter_territory": "Nombre de trajets inter-territoires",
    "share_journeys_intra_territory": "% de trajets intra-territoire",
    "share_journeys_inter_territory": "% de trajets inter-territoires",
    "share_drivers": "% des conducteurs",
    "num_trips": "Nombre de trips",
    "is_intra_driver": "Conducteur intra",
    "driver_campaign_type": "Type de campagne du conducteur",
    "passenger_campaign_type": "Type de campagne du passager",
    "drivers_share": "% des conducteurs",
    "week_number": "Semaine n°",
    "passengers_share": "% des passagers",
    "num_passenger": "Nombre de passager",
    "is_near_station_fmt": "Catégorie de distance à une gare",
    "has_direct_train_line": "Possède une ligne TC directe",
    "name": "Nom",
    "amount_aom_avg" : "Incitation AOM moyenne",
    "line_name_end":"Ligne",
    "amount_aom": "Incitation AOM",
    "incentive_amount": "Incitation totale",
    "variable": "",
}

import plotly.io as pio

pio.templates.default = "simple_white"
pio.templates[pio.templates.default].layout.font.size = 16

## duckdb


In [ ]:
conn = duckdb.connect(
    "db.duckdb",
    config={"memory_limit": "16GiB", "threads": 4, "preserve_insertion_order": False},
)
%sql conn --alias duckdb

In [ ]:
%%sql
INSTALL spatial;

LOAD spatial;

# Queries


In [ ]:
SQL_ENGINE = create_engine(DB_URL)

## Journeys


In [ ]:
df_journeys_raw = pl.read_parquet("df_journeys_raw_idfm.parquet")

## Opérateurs


In [ ]:
df_operators = pl.read_database(
    query="""
SELECT
    "_id",
    "name",
    "siret"
from operator.operators
where deleted_at is null
and name!='BlaBlaCar'
""",
    connection=SQL_ENGINE,
)

In [ ]:
df_karos=pl.DataFrame({"_id":[999], "name":["Karos"], "siret":["80279897500024"]})
df_operators = pl.concat([df_operators, df_karos])
df_operators

# Reseau IDFM


In [ ]:
%%sql
CREATE TABLE
  IF NOT EXISTS gares_idfm AS
SELECT
  id_gares,
  nom_gares,
  nom_so_gar,
  nom_su_gar,
  id_ref_zdc,
  nom_zdc,
  id_ref_zda,
  nom_zda,
  idrefliga,
  idrefligc,
  res_com,
  indice_lig,
  mode,
  tertrain,
  terrer,
  termetro,
  tertram,
  terval,
  exploitant,
  idf,
  ST_FlipCoordinates (geom) AS geom --  EPSG:4326 coordinate system (WGS84), with [latitude, longitude] axis order
FROM
    ST_Read('{{IDF_GEOJSON}}')
WHERE
  mode IN ('TRAIN', 'RER');

In [ ]:
%%sql
SELECT
  COUNT(*)
FROM
  gares_idfm;

In [ ]:
%%sql
CREATE INDEX IF NOT EXISTS gares_geom_index ON gares_idfm USING RTREE (geom);

In [ ]:
%%sql
FROM
  (DESCRIBE gares_idfm);

In [ ]:
%%sql df_idfm_stations <<
SELECT
    *,
    ST_asText(geom) as geom_wkt
FROM gares_idfm

In [ ]:
df_idfm_stations

# Identification incitateurs


In [ ]:
df_journeys_raw = df_journeys_raw.with_columns(
    pl.col("incentive_sirets").list.contains(AOM_SIRET).alias("incentived_by_aom"),
    (
        pl.col("incentive_sirets")
        .list.set_intersection(df_operators["siret"].to_list())
        .list.len()
        > 0
    ).alias("incentived_by_operator"),
)

# Traitements geo


In [ ]:
df_journeys_raw = df_journeys_raw.with_columns(
        pl.col("start_position")
        .map_elements(lambda x: shapely.from_wkb(x).wkt, return_dtype=pl.String)
        .alias("start_pos"),
        pl.col("end_position")
        .map_elements(lambda x: shapely.from_wkb(x).wkt, return_dtype=pl.String)
        .alias("end_pos"),
        pl.col("start_position")
        .map_elements(lambda x: shapely.from_wkb(x).y, return_dtype=pl.Float64)
        .alias("start_latitude"),
        pl.col("start_position")
        .map_elements(lambda x: shapely.from_wkb(x).x, return_dtype=pl.Float64)
        .alias("start_longitude"),
            pl.col("end_position")
        .map_elements(lambda x: shapely.from_wkb(x).y, return_dtype=pl.Float64)
        .alias("end_latitude"),
        pl.col("end_position")
        .map_elements(lambda x: shapely.from_wkb(x).x, return_dtype=pl.Float64)
        .alias("end_longitude"),
    )

# Filtrage des journeys sans incitations


In [ ]:
df_journeys = df_journeys_raw.filter((pl.col("incentive_amount") > 0))

# Création de la table des journeys sur duckdb


In [ ]:
%%sql
CREATE TABLE
  if NOT EXISTS journeys_raw AS
SELECT
  _id,
  operator_id,
  operator_journey_id,
  operator_trip_id,
  driver_identity_key,
  first_trip_datetime,
  passenger_identity_key,
  passenger_first_trip_datetime,
  start_datetime,
  end_datetime,
  distance,
  driver_revenue,
  passenger_contribution,
  incentive_amount,
  amount_aom,
  incentive_sirets,
  start_position,
  end_position,
  passenger_seats,
  is_fully_inside_campaign_area,
  journey_line,
  start_com,
  end_com,
  incentived_by_aom,
  incentived_by_operator,
  ST_FlipCoordinates (ST_GeomFromText (start_pos)) AS start_pos,
  ST_FlipCoordinates (ST_GeomFromText (end_pos)) AS end_pos
FROM
  df_journeys_raw

In [ ]:
%%sql
CREATE INDEX IF NOT EXISTS start_pos_idx ON journeys_raw USING RTREE (start_pos);

CREATE INDEX IF NOT EXISTS end_pos_idx ON journeys_raw USING RTREE (end_pos);

In [ ]:
%%sql
FROM
  (DESCRIBE journeys_raw)

In [ ]:
%%sql
SELECT
  *
FROM
  journeys_raw
LIMIT
  5

In [ ]:
%%sql
SELECT
  COUNT(*)
FROM
  journeys_raw

In [ ]:
incentived_trip_filter_expr = pl.col("incentive_amount") > 0

agg_expressions = [
    pl.col("_id").n_unique().alias("num_journeys"),
    pl.col("_id")
    .filter(pl.col("is_fully_inside_campaign_area"))
    .n_unique()
    .alias("num_journeys_intra_territory"),
    pl.col("_id")
    .filter(pl.col("is_fully_inside_campaign_area") & incentived_trip_filter_expr)
    .n_unique()
    .alias("num_journeys_intra_territory_incentived_trips"),
    pl.col("_id")
    .filter(incentived_trip_filter_expr)
    .n_unique()
    .alias("num_journeys_incentived"),
    pl.col("_id")
    .filter(pl.col("incentived_by_aom"))
    .n_unique()
    .alias("num_journeys_with_aom_incentive"),
    pl.col("_id")
    .filter(pl.col("incentived_by_operator"))
    .n_unique()
    .alias("num_journeys_with_operator_incentive"),
    pl.col("_id")
    .filter(pl.col("is_fully_inside_campaign_area"))
    .n_unique()
    .alias("num_journeys_intra"),
    (pl.col("distance") / 1000).mean().alias("distance_avg"),
    (pl.col("distance").filter(incentived_trip_filter_expr) / 1000)
    .mean()
    .alias("distance_incentived_trips_avg"),
    (pl.col("amount_aom").sum()/100).alias("amount_aom_sum"),
    (pl.col("amount_aom").mean()/100).alias("amount_aom_avg"),
    (pl.col("incentive_amount").mean() / 100).alias("incentive_amount_avg"),
    (pl.col("passenger_contribution").filter(incentived_trip_filter_expr.not_()) / 100)
    .mean()
    .alias("passenger_contribution_avg"),
    (pl.col("passenger_contribution").filter(incentived_trip_filter_expr) / 100)
    .mean()
    .alias("passenger_contribution_incentived_trips_avg"),
    (
        pl.col("driver_revenue").filter(incentived_trip_filter_expr.not_()).mean() / 100
    ).alias("driver_revenue_avg"),
    (pl.col("driver_revenue").filter(incentived_trip_filter_expr).mean() / 100).alias(
        "driver_revenue_incentived_trips_avg"
    ),
    (
        pl.col("incentive_amount")
        .filter(pl.col("is_fully_inside_campaign_area"))
        .mean()
        / 100
    ).alias("incentive_amount_intra_avg"),
    (
        pl.col("passenger_contribution")
        .filter(pl.col("is_fully_inside_campaign_area") & incentived_trip_filter_expr)
        .mean()
        / 100
    ).alias("passenger_contribution_intra_avg"),
    (
        pl.col("driver_revenue")
        .filter(pl.col("is_fully_inside_campaign_area") & incentived_trip_filter_expr)
        .mean()
        / 100
    ).alias("driver_revenue_intra_avg"),
    (
        pl.col("incentive_amount")
        .filter(
            pl.col("is_fully_inside_campaign_area").not_() & incentived_trip_filter_expr
        )
        .mean()
        / 100
    ).alias("incentive_amount_inter_avg"),
    (
        pl.col("passenger_contribution")
        .filter(
            pl.col("is_fully_inside_campaign_area").not_() & incentived_trip_filter_expr
        )
        .mean()
        / 100
    ).alias("passenger_contribution_inter_avg"),
    (
        pl.col("driver_revenue")
        .filter(
            pl.col("is_fully_inside_campaign_area").not_() & incentived_trip_filter_expr
        )
        .mean()
        / 100
    ).alias("driver_revenue_inter_avg"),
    (10 * (pl.col("incentive_amount") / pl.col("distance")))
    .mean()
    .alias("incentive_amount_per_km_avg"),
    (10 * (pl.col("amount_aom") / pl.col("distance")).filter(
            incentived_trip_filter_expr
        ))
    .mean()
    .alias("aom_amount_per_km_avg"),    
    (
        10
        * (pl.col("passenger_contribution") / pl.col("distance")).filter(
            incentived_trip_filter_expr
        )
    )
    .mean()
    .alias("passenger_contribution_per_km_avg"),
    (
        10
        * (pl.col("passenger_contribution") / pl.col("distance")).filter(
            incentived_trip_filter_expr
        )
    )
    .mean()
    .alias("passenger_contribution_per_km_incentived_trips_avg"),
    (
        10
        * (pl.col("driver_revenue") / pl.col("distance")).filter(
            incentived_trip_filter_expr
        )
    )
    .mean()
    .alias("driver_revenue_per_km_avg"),
    (
        10
        * (pl.col("driver_revenue") / pl.col("distance")).filter(
            incentived_trip_filter_expr.not_() &  pl.col("is_fully_inside_campaign_area").not_()
        )
    )
    .mean()
    .alias("driver_revenue_per_km_avg_inter"),
     (
        10
        * (pl.col("driver_revenue") / pl.col("distance")).filter(
            incentived_trip_filter_expr.not_() &  pl.col("is_fully_inside_campaign_area")
        )
    )
    .mean()
    .alias("driver_revenue_per_km_avg_intra"),
    (
        10
        * (pl.col("driver_revenue") / pl.col("distance")).filter(
            incentived_trip_filter_expr
        )
    )
    .mean()
    .alias("driver_revenue_per_km_incentived_trips_avg"),
    (
        10
        * (pl.col("driver_revenue") / pl.col("distance")).filter(
            incentived_trip_filter_expr &  pl.col("is_fully_inside_campaign_area").not_()
        )
    )
    .mean()
    .alias("driver_revenue_per_km_incentived_trips_avg_inter"),
    (
        10
        * (pl.col("driver_revenue") / pl.col("distance")).filter(
            incentived_trip_filter_expr &  pl.col("is_fully_inside_campaign_area")
        )
    )
    .mean()
    .alias("driver_revenue_per_km_incentived_trips_avg_intra"),
    pl.col("driver_identity_key").n_unique().alias("number_of_unique_driver"),
    pl.col("passenger_identity_key").n_unique().alias("number_of_unique_passenger"),

]

In [ ]:
df_stats_by_month = (
    df_journeys_raw.group_by(pl.col("start_datetime").dt.truncate("1mo").alias("month"))
    .agg(agg_expressions)
    .sort(pl.col("month"))
)

In [ ]:
df_stats_by_week = (
    df_journeys_raw.filter(
        pl.col("start_datetime") <= datetime(2025, 7, 31, tzinfo=ZoneInfo("GMT"))
    )
    .group_by(pl.col("start_datetime").dt.truncate("1w").alias("week"))
    .agg(agg_expressions)
    .sort(pl.col("week"))
)

df_stats_by_week_rm_citygo = (
    df_journeys_raw.filter(
        pl.col("start_datetime") <= datetime(2025, 7, 31, tzinfo=ZoneInfo("GMT")), pl.col("operator_id") != 274
    )
    .group_by(pl.col("start_datetime").dt.truncate("1w").alias("week"))
    .agg(agg_expressions)
    .sort(pl.col("week"))
)

# Autour des gares


## Transformations spatiales


In [ ]:
%%sql
FROM
  (DESCRIBE gares_idfm)

In [ ]:
conn.sql(
    """
create or replace table journeys_raw_with_stations_start as (
SELECT
      jr."_id",
      jr.start_pos,
      jr.end_pos,
      g.id_gares as id_gares_start,
      g.nom_gares as nom_gares_start,
      g.mode as mode_start,
      g.geom as geom_start,
      ST_DISTANCE(jr.start_pos,g.geom) as distance_to_station_start,
      ST_DISTANCE_SPHERE(jr.start_pos,g.geom) as distance_to_station_sphere_start,
      ST_Distance_Spheroid(jr.start_pos,g.geom) as distance_to_station_spheroid_start
  FROM journeys_raw jr
  left join gares_idfm g on ST_DWithin(jr.start_pos,g.geom,0.1)
)

    """
)

In [ ]:
conn.sql(
    """
create or replace table journeys_raw_with_nearest_stations_start as (
SELECT
      *
FROM journeys_raw_with_stations_start
qualify (row_number() over (partition by "_id" order by distance_to_station_sphere_start asc nulls last))=1
)
    """
)

In [ ]:
%%sql
DROP TABLE journeys_raw_with_stations_start

In [ ]:
%sql CHECKPOINT

In [ ]:
%%sql
SELECT
  COUNT(*)
FROM
  journeys_raw_with_nearest_stations_start

In [ ]:
%%sql
FROM
  (DESCRIBE journeys_raw_with_nearest_stations_start)

In [ ]:
conn.sql(
    """
create or replace table journeys_raw_with_stations_end as (
SELECT
      jr.*,
      g.id_gares as id_gares_end,
      g.nom_gares as nom_gares_end,
      g.mode as mode_end,
      g.geom as geom_end,
      ST_DISTANCE(jr.end_pos,g.geom) as distance_to_station_end,
      ST_DISTANCE_SPHERE(jr.end_pos,g.geom) as distance_to_station_sphere_end,
      ST_Distance_Spheroid(jr.end_pos,g.geom) as distance_to_station_spheroid_end
  FROM journeys_raw_with_nearest_stations_start jr
  left join gares_idfm g on ST_DWithin(jr.end_pos,g.geom,0.1)
)
"""
)

In [ ]:
%%sql
SELECT
  COUNT(*)
FROM
  journeys_raw_with_stations_end

In [ ]:
%%sql
DROP TABLE journeys_raw_with_nearest_stations_start

In [ ]:
%%sql
checkpoint

In [ ]:
conn.sql(
    """
create or replace table journeys_raw_with_nearest_stations as (
SELECT
      *
FROM journeys_raw_with_stations_end
qualify (row_number() over (partition by "_id" order by distance_to_station_sphere_end asc nulls last))=1
)
    """
)

In [ ]:
%%sql 
DROP TABLE journeys_raw_with_stations_end

In [ ]:
%%sql
SELECT
  COUNT(*)
FROM
  journeys_raw_with_nearest_stations

In [ ]:
%%sql
FROM
  (DESCRIBE journeys_raw_with_nearest_stations)

In [ ]:
%%sql df_journeys_raw_with_nearest_stations <<
SELECT
  *,
  ST_AsText(start_pos) as start_pos_wkt,
  ST_AsText(end_pos) as end_pos_wkt,
  ST_AsText (geom_start) AS geom_start_wkt,
  ST_AsText (geom_end) AS geom_end_wkt
FROM
  journeys_raw_with_nearest_stations

In [ ]:
df_journeys_raw_with_nearest_stations = (
    df_journeys_raw_with_nearest_stations.with_columns(
        pl.selectors.starts_with("distance_to_station").fill_null(float("+inf"))
    )
)

In [ ]:

def get_osrm_distance(coord1, coord2, osrm_server= "http://127.0.0.1:5000", max_retries=10, base_delay=5):
    from shapely import from_wkt

    try:
        coord1 = from_wkt(coord1)
        coord2 = from_wkt(coord2)
        if coord1 is None or coord2 is None:
            print(f"Parsing WKT échoué - coord1: {coord1}, coord2: {coord2}")
            return None
            
        if not hasattr(coord1, 'x') or not hasattr(coord1, 'y') or \
           not hasattr(coord2, 'x') or not hasattr(coord2, 'y'):
            print(f"Coordonnées invalides - coord1: {type(coord1)}, coord2: {type(coord2)}")
            return None
            
    except Exception as e:
        print(f"Erreur de parsing des coordonnées {coord1} -> {coord2}: {e}")
        return None
    
    # Format: longitude,latitude;longitude,latitude
    url = f"{osrm_server}/route/v1/walking/{coord1.y},{coord1.x};{coord2.y},{coord2.x}"
    params = {
        'overview': 'false',  
        'steps': 'false'      
    }
    
    for attempt in range(max_retries + 1):
        try:
            response = requests.get(url, params=params, timeout=5)
            response.raise_for_status()
            data = response.json()
            
            if len(data['routes']) > 0:
                # Distance en mètres
                return data['routes'][0]['distance']
            else:
                print(f"Aucune route trouvée pour {coord1.y},{coord1.x} -> {coord2.y},{coord2.x}")
                return None
                
        except requests.exceptions.Timeout:
            if attempt < max_retries:
                delay = base_delay * (2 ** attempt)  # Backoff exponentiel
                print(f"Timeout OSRM (tentative {attempt + 1}/{max_retries + 1}). Retry dans {delay}s...")
                time.sleep(delay)
                continue
            else:
                print(f"Timeout définitif OSRM pour {coord1.y},{coord1.x} -> {coord2.y},{coord2.x}")
                return None
                
        except requests.exceptions.ConnectionError:
            if attempt < max_retries:
                delay = base_delay * (2 ** attempt)
                print(f"Erreur de connexion OSRM (tentative {attempt + 1}/{max_retries + 1}). Retry dans {delay}s...")
                time.sleep(delay)
                continue
            else:
                print(f"Erreur de connexion définitive OSRM pour {coord1.y},{coord1.x} -> {coord2.y},{coord2.x}")
                return None
                
        except requests.exceptions.HTTPError as e:
            if 400 <= e.response.status_code < 500:
                print(f"Erreur HTTP {e.response.status_code} OSRM pour {coord1.y},{coord1.x} -> {coord2.y},{coord2.x}: {e}")
                return None
            else:
                if attempt < max_retries:
                    delay = base_delay * (2 ** attempt)
                    print(f"Erreur serveur OSRM {e.response.status_code} (tentative {attempt + 1}/{max_retries + 1}). Retry dans {delay}s...")
                    time.sleep(delay)
                    continue
                else:
                    print(f"Erreur serveur définitive OSRM pour {coord1.y},{coord1.x} -> {coord2.y},{coord2.x}: {e}")
                    return None
                    
        except requests.exceptions.RequestException as e:
            if attempt < max_retries:
                delay = base_delay * (2 ** attempt)
                print(f"Erreur requête OSRM (tentative {attempt + 1}/{max_retries + 1}). Retry dans {delay}s...")
                time.sleep(delay)
                continue
            else:
                print(f"Erreur requête définitive OSRM pour {coord1.y},{coord1.x} -> {coord2.y},{coord2.x}: {e}")
                return None
                
        except Exception as e:
            print(f"Erreur inattendue OSRM pour {coord1.y},{coord1.x} -> {coord2.y},{coord2.x}: {e}")
            return None
    
    return None


In [ ]:
def add_osrm_distances(df, osrm_server = "http://localhost:5000", batch_size = 100):

    def process_batch_start(batch_series):
        """Traite un batch pour les distances start"""
    results_start = []
    results_end = []
    
    total = len(df)
    for i, row in enumerate(df.iter_rows(named=True)):
        
        # Distance start
        dist_start = get_osrm_distance(
            row['start_pos_wkt'], 
            row['geom_start_wkt'], 
            osrm_server
        )
        results_start.append(dist_start)
        
        # Distance end  
        dist_end = get_osrm_distance(
            row['end_pos_wkt'], 
            row['geom_end_wkt'], 
            osrm_server
        )
        results_end.append(dist_end)
        
        # Progress + pause
        if i % 10 == 0:
            print(f"Progress: {i}/{total} ({i/total*100:.1f}%)")
        
    
    return df.with_columns([
        pl.Series("distance_to_station_osrm_start", results_start),
        pl.Series("distance_to_station_osrm_end", results_end)
    ])

In [ ]:
if USE_CACHED_JOURNEYS_WITH_NEAREST_STATION:
    df_journeys_raw_with_nearest_stations = pl.read_parquet("df_journeys_raw_with_nearest_stations_osrm.parquet")
else:
    df_journeys_raw_with_nearest_stations_osrm = add_osrm_distances(df_journeys_raw_with_nearest_stations)
    df_journeys_raw_with_nearest_stations_osrm.head()
    df_journeys_raw_with_nearest_stations_osrm.write_parquet("df_journeys_raw_with_nearest_stations_osrm.parquet", compression_level=6)
    

## Distribution des distances à la gare la plus proche


### Point de départ


In [ ]:
distance_var = None
if "distance_to_station_osrm_start" in df_journeys_raw_with_nearest_stations.columns:
    distance_var = "distance_to_station_osrm"
else:
    distance_var = "distance_to_station_spheroid"

print(distance_var)

In [ ]:
breaks = [100, 200, 500] + list(range(1000, 11000, 1000))

df_journeys_count_by_distance_to_nearest_station_start = (
    df_journeys_raw_with_nearest_stations.group_by(
        pl.col(distance_var + "_start").cut(
            breaks=breaks, include_breaks=True
        )
    )
    .agg(pl.col("_id").n_unique().alias("num_journeys"))
    .with_columns(
        (pl.col("num_journeys") / pl.col("num_journeys").sum()).alias("share_journeys"),
        pl.col(distance_var + "_start").struct.unnest(),
    )
    .with_columns(
        pl.format("{}%", (100 * pl.col("share_journeys")).round(2)).alias(
            "share_journeys_fmt"
        )
    )
    .sort("category")
)

In [ ]:
px.bar(
    df_journeys_count_by_distance_to_nearest_station_start,
    x="category",
    y="num_journeys",
    text="share_journeys_fmt",
    labels={**labels_map, "category": "Catégorie de distance (en mètres)"},
    template="simple_white",
    title="Distribution du nombre de trajets en fonction de la distance à la gare RER/Transilien la plus proche"
    "<br><sub>Par rapport au point de départ du trajet</sub>",
)

### Point d'arrivée


In [ ]:
df_journeys_count_by_distance_to_nearest_station_end = (
    df_journeys_raw_with_nearest_stations.group_by(
        pl.col(distance_var + "_end").cut(
            breaks=[100, 200, 500] + list(range(1000, 11000, 1000)), include_breaks=True
        )
    )
    .agg(pl.col("_id").n_unique().alias("num_journeys"))
    .with_columns(
        (pl.col("num_journeys") / pl.col("num_journeys").sum()).alias("share_journeys"),
        pl.col(distance_var + "_end").struct.unnest(),
    )
    .with_columns(
        pl.format("{}%", (100 * pl.col("share_journeys")).round(2)).alias(
            "share_journeys_fmt"
        )
    )
    .sort("category")
)

In [ ]:
px.bar(
    df_journeys_count_by_distance_to_nearest_station_end,
    x="category",
    y="num_journeys",
    text="share_journeys_fmt",
    labels={**labels_map, "category": "Catégorie de distance (en mètres)"},
    template="simple_white",
    title="Distribution du nombre de trajets en fonction de la distance à la gare RER/Transilien la plus proche"
    "<br><sub>Par rapport au point d'arrivée du trajet</sub>",
)

## Origine et destination


In [ ]:
breaks = [100, 200, 500] + list(range(1000, 11000, 1000))
df_journeys_count_by_distance_to_nearest_station = (
    df_journeys_raw_with_nearest_stations.with_columns(
        pl.max_horizontal(
            [distance_var + "_start", distance_var + "_end"]
        ).alias("max_distance_to_nearest_station")
    )
    .group_by(
        pl.col("max_distance_to_nearest_station").cut(
            breaks=[100, 200, 500] + list(range(1000, 11000, 1000)), include_breaks=True
        )
    )
    .agg(pl.col("_id").n_unique().alias("num_journeys"))
    .with_columns(
        (pl.col("num_journeys") / pl.col("num_journeys").sum()).alias("share_journeys"),
        pl.col("max_distance_to_nearest_station").struct.unnest(),
    )
    .with_columns(
        pl.format("{}%", (100 * pl.col("share_journeys")).round(2)).alias(
            "share_journeys_fmt"
        )
    )
    .sort("category")
)

In [ ]:
fig_journeys_count_by_distance_to_nearest_station_OandD = px.bar(
    df_journeys_count_by_distance_to_nearest_station,
    x="category",
    y="num_journeys",
    text="share_journeys_fmt",
    labels={**labels_map, "category": "Catégorie de distance (en mètres)"},
    template="simple_white",
    title="Distribution du nombre de trajets en fonction de la distance à la gare RER/Transilien la plus proche"
    "<br><sub>Par rapport au point de départ ET d'arrivée du trajet</sub>",
)

fig_journeys_count_by_distance_to_nearest_station_OandD.show()

fig_journeys_count_by_distance_to_nearest_station_OandD.write_html(
    "outputs_idfm/fig_journeys_count_by_distance_to_nearest_station_OandD.html"
)
fig_journeys_count_by_distance_to_nearest_station_OandD.write_image(
    "outputs_idfm/fig_journeys_count_by_distance_to_nearest_station_OandD.svg", width=1280, height=720
)

### Multi point


In [ ]:
df_journeys_count_by_distance_to_nearest_stations = (
    df_journeys_raw_with_nearest_stations.group_by(
        pl.col(distance_var + "_start").cut(
            breaks=[100, 200, 500] + list(range(1000, 11000, 1000)), include_breaks=True
        )
    )
    .agg(pl.col("_id").n_unique().alias("num_journeys"))
    .with_columns(
        (pl.col("num_journeys") / pl.col("num_journeys").sum()).alias("share_journeys"),
        pl.col(distance_var + "_start").struct.unnest(),
    )
    .with_columns(
        pl.format("{}%", (100 * pl.col("share_journeys")).round(2)).alias(
            "share_journeys_fmt"
        )
    )
    .sort("category")
)

In [ ]:
px.scatter(
    df_journeys_raw_with_nearest_stations.group_by(
        (
            pl.col(distance_var + "_start").fill_null(float("inf")) / 100
        ).round(0),
        (
            pl.col(distance_var + "_end").fill_null(float("inf")) / 100
        ).round(0),
    ).agg(pl.col("_id").n_unique().alias("num_journeys")),
    x=distance_var + "_start",
    y=distance_var + "_end",
    size="num_journeys",
    template="simple_white",
)

## Top des gares


### Au point de départ


In [ ]:
df_top_nearest_stations_start = (
    df_journeys_raw_with_nearest_stations.filter(
        pl.col(distance_var + "_start").is_finite()
    )
    .group_by("nom_gares_start")
    .agg(
        pl.col("_id").n_unique().alias("num_journeys"),
        pl.col("geom_start_wkt").max(),
        pl.col(distance_var + "_start").mean(),
    )
    .with_columns(
        (pl.col("num_journeys") / pl.col("num_journeys").sum()).alias("share_journeys")
    )
    .with_columns(
        pl.format(
            "{}%<br>{}m",
            (100 * pl.col("share_journeys")).round(2),
            pl.col(distance_var + "_start").round(0).cast(pl.Int64),
        ).alias("share_journeys_fmt")
    )
    .sort("num_journeys", descending=True)
    .drop_nulls()
)

df_top_nearest_stations_end = (
    df_journeys_raw_with_nearest_stations.filter(
        pl.col(distance_var + "_end").is_finite()
    )
    .group_by("nom_gares_end")
    .agg(
        pl.col("_id").n_unique().alias("num_journeys"),
        pl.col("geom_start_wkt").max(),
        pl.col(distance_var + "_end").mean(),
    )
    .with_columns(
        (pl.col("num_journeys") / pl.col("num_journeys").sum()).alias("share_journeys")
    )
    .with_columns(
        pl.format(
            "{}%<br>{}m",
            (100 * pl.col("share_journeys")).round(2),
            pl.col(distance_var + "_end").round(0).cast(pl.Int64),
        ).alias("share_journeys_fmt")
    )
    .sort("num_journeys", descending=True)
    .drop_nulls()
)

In [ ]:
fig_station_by_num_near_journeys= px.bar(
    df_top_nearest_stations_start.head(10),
    x="nom_gares_start",
    y="num_journeys",
    text="share_journeys_fmt",
    template="simple_white",
    labels=labels_map,
    title="TOP 10 des gares par nombre de trajets démarrant à proximité"
    "<br><sub>Dans les barres sont affichées la proportion au regard du total des trajets et la distance moyenne au départ.</sub>",
    height=500,
)

fig_station_by_num_near_journeys.show()

fig_station_by_num_near_journeys.write_html(
    "outputs_idfm/fig_station_by_num_near_journeys.html"
)
fig_station_by_num_near_journeys.write_image(
    "outputs_idfm/fig_station_by_num_near_journeys.svg", width=1280, height=720
)

In [ ]:
fig_station_by_num_near_journeys_end= px.bar(
    df_top_nearest_stations_end.head(10),
    x="nom_gares_end",
    y="num_journeys",
    text="share_journeys_fmt",
    template="simple_white",
    labels=labels_map,
    title="TOP 10 des gares par nombre de trajets finissant à proximité"
    "<br><sub>Dans les barres sont affichées la proportion au regard du total des trajets et la distance moyenne au départ.</sub>",
    height=500,
)

fig_station_by_num_near_journeys_end.show()

fig_station_by_num_near_journeys_end.write_html(
    "outputs_idfm/fig_station_by_num_near_journeys_end.html"
)
fig_station_by_num_near_journeys_end.write_image(
    "outputs_idfm/fig_station_by_num_near_journeys_end.svg", width=1280, height=720
)

## À moins d'un kilometre


In [ ]:
df_top_nearest_stations_start_1km = (
    df_journeys_raw_with_nearest_stations.filter(
        pl.col(distance_var + "_start") <= 1000
    )
    .group_by("nom_gares_start")
    .agg(
        pl.col("_id").n_unique().alias("num_journeys"),
        pl.col("geom_start_wkt").max(),
        pl.col(distance_var + "_start").mean(),
    )
    .with_columns(
        (pl.col("num_journeys") / pl.col("num_journeys").sum()).alias("share_journeys")
    )
    .with_columns(
        pl.format(
            "{}%<br>{}m",
            (100 * pl.col("share_journeys")).round(2),
            pl.col(distance_var + "_start").round(0).cast(pl.Int64),
        ).alias("share_journeys_fmt")
    )
    .sort("num_journeys", descending=True)
    .drop_nulls()
)

df_top_nearest_stations_end_1km = (
    df_journeys_raw_with_nearest_stations.filter(
        pl.col(distance_var + "_end") <= 1000
    )
    .group_by("nom_gares_end")
    .agg(
        pl.col("_id").n_unique().alias("num_journeys"),
        pl.col("geom_start_wkt").max(),
        pl.col(distance_var + "_end").mean(),
    )
    .with_columns(
        (pl.col("num_journeys") / pl.col("num_journeys").sum()).alias("share_journeys")
    )
    .with_columns(
        pl.format(
            "{}%<br>{}m",
            (100 * pl.col("share_journeys")).round(2),
            pl.col(distance_var + "_end").round(0).cast(pl.Int64),
        ).alias("share_journeys_fmt")
    )
    .sort("num_journeys", descending=True)
    .drop_nulls()
)

In [ ]:
fig_station_by_num_near_journeys_closer_than_one_km = px.bar(
    df_top_nearest_stations_start_1km.head(10),
    x="nom_gares_start",
    y="num_journeys",
    text="share_journeys_fmt",
    template="simple_white",
    labels=labels_map,
    title="TOP 10 des gares par nombre de trajets démarrant à moins d'un km"
    "<br><sub>Dans les barres sont affichées la proportion au regard du total des trajets et la distance moyenne au départ.</sub>",
    height=500,
)

fig_station_by_num_near_journeys_closer_than_one_km.show()

fig_station_by_num_near_journeys_closer_than_one_km.write_html(
    "outputs_idfm/fig_station_by_num_near_journeys_closer_than_one_km.html"
)
fig_station_by_num_near_journeys_closer_than_one_km.write_image(
    "outputs_idfm/fig_station_by_num_near_journeys_closer_than_one_km.svg", width=1280, height=720
)



In [ ]:
fig_station_by_num_near_journeys_closer_than_one_km_end = px.bar(
    df_top_nearest_stations_end_1km.head(10),
    x="nom_gares_end",
    y="num_journeys",
    text="share_journeys_fmt",
    template="simple_white",
    labels=labels_map,
    title="TOP 10 des gares par nombre de trajets finissant à moins d'un km"
    "<br><sub>Dans les barres sont affichées la proportion au regard du total des trajets et la distance moyenne au départ.</sub>",
    height=500,
)

fig_station_by_num_near_journeys_closer_than_one_km_end.show()

fig_station_by_num_near_journeys_closer_than_one_km_end.write_html(
    "outputs_idfm/fig_station_by_num_near_journeys_closer_than_one_km_end.html"
)
fig_station_by_num_near_journeys_closer_than_one_km_end.write_image(
    "outputs_idfm/fig_station_by_num_near_journeys_closer_than_one_km_end.svg", width=1280, height=720
)



## Analyse des trajets avec O et D à proximité d'une gare


In [ ]:
MAX_DISTANCE = 1000  # 1 km

In [ ]:
df_journeys_raw_with_nearest_stations_full = (
    df_journeys_raw.join(
        df_journeys_raw_with_nearest_stations.filter(
            pl.max_horizontal(
                distance_var + "_start", distance_var + "_end"
            )
            <= MAX_DISTANCE
        ),
        on="_id",
        how="left",
        validate="1:1",
        coalesce=False,
    )
    .with_columns(pl.col("_id_right").is_not_null().alias("is_near_station"))
    .with_columns(
        pl.when(pl.col("is_near_station"))
        .then(pl.lit("Trajets proches d'une gare"))
        .otherwise(pl.lit("Trajets éloignés d'une gare"))
        .alias("is_near_station_fmt")
    )
)
df_journeys_raw_with_nearest_stations_full.shape

In [ ]:
distance_var

In [ ]:
with pl.Config(set_fmt_str_lengths=120, set_tbl_width_chars=1000):
    print(
        df_journeys_raw_with_nearest_stations_full.filter(pl.col("is_near_station")==True)
        .select(
            pl.col("_id").n_unique().alias("Nombre de journeys"),
            pl.col("_id")
            .filter(pl.col("incentive_amount") > 0)
            .n_unique()
            .alias("Nombre de journeys avec incitation"),
            (
                100
                * pl.col("_id").filter(pl.col("incentive_amount") > 0).n_unique()
                / pl.col("_id").n_unique()
            ).alias("% journeys avec incitation"),
            pl.col("_id")
            .filter(pl.col("incentived_by_aom"))
            .n_unique()
            .alias("Nombre de journeys avec incitation AOM"),
            (
                100
                * pl.col("_id").filter(pl.col("incentived_by_aom")).n_unique()
                / pl.col("_id").n_unique()
            ).alias("% journeys avec incitation AOM"),
            pl.col("_id")
            .filter(pl.col("incentived_by_operator"))
            .n_unique()
            .alias("Nombre de journeys avec incitation opérateur"),
            (
                100
                * pl.col("_id").filter(pl.col("incentived_by_operator")).n_unique()
                / pl.col("_id").n_unique()
            ).alias("% journeys avec incitation opérateur"),
            (
                100
                * pl.col("_id").filter(pl.col("incentived_by_operator"), ~pl.col("incentived_by_aom")).n_unique()
                / pl.col("_id").n_unique()
            ).alias("% trajets avec incitation opérateur seule"),
        )
        .with_columns(pl.selectors.all().round(2))
        .unpivot()
    )

In [ ]:
df_stats_by_month_full = (
    df_journeys_raw_with_nearest_stations_full.group_by(
        [
            pl.col("start_datetime").dt.truncate("1mo").alias("month"),
            "is_near_station_fmt",
        ]
    )
    .agg(agg_expressions)
    .sort(pl.col("month"))
)

In [ ]:
df_stats_by_week_filtered = (
    df_journeys_raw_with_nearest_stations_full.filter(
        pl.col("start_datetime") <= datetime(2025, 7, 31, tzinfo=ZoneInfo("GMT"))
    )
    .group_by(
        [
            pl.col("start_datetime").dt.truncate("1w").alias("week"),
            "is_near_station_fmt",
        ]
    )
    .agg(agg_expressions)
    .sort(pl.col("week"))
)

### Evolution


#### Globale


In [ ]:
colors_map = {
    "Trajets éloignés d'une gare": "#ffae2b",
    "Trajets proches d'une gare": "#2B7CFF",
}
category_order = ["Trajets proches d'une gare", "Trajets éloignés d'une gare"]


In [ ]:
fig_journeys_by_month_near_vs_far = px.line(
    df_stats_by_month_full,
    x="month",
    y="num_journeys_incentived",
    color="is_near_station_fmt",
    color_discrete_map=colors_map,
    template="simple_white",
    labels=labels_map,
    log_y=True,
    title="Nombre de journeys par mois - "
    f'<br><sub>Les trajets dits "proches" sont ceux avec O/D à moins de {MAX_DISTANCE / 1000:.0f} km d\'une gare<sub>',
)
fig_journeys_by_month_near_vs_far.show()

fig_journeys_by_month_near_vs_far
# fig_journeys_by_month_near_vs_far.update_yaxes(
#     range=[0, df_stats_by_month_full["num_journeys_incentived"].max() - 1.1]
# )

fig_journeys_by_month_near_vs_far.write_html(
    OUTPUT_PATH / "fig_journeys_par_mois_near_vs_far.html"
)
fig_journeys_by_month_near_vs_far.write_image(
    OUTPUT_PATH / "fig_journeys_par_mois_near_vs_far.svg", width=1280, height=720
)

#### Opérateur incitateurs


In [ ]:
fig_journeys_by_operator_filtered = px.line(
    df_journeys_raw_with_nearest_stations_full.filter(pl.col("is_near_station"))
    .explode("incentive_sirets")
    .join(
        df_operators,
        left_on="incentive_sirets",
        right_on="siret",
        how="left",
        suffix="_operators",
    )
    .group_by(["name", pl.col("start_datetime").dt.truncate("1mo")])
    .agg(pl.col("operator_journey_id").n_unique().alias("num_journeys"))
    .rename({"name": "operator", "start_datetime": "month"})
    .sort("month"),
    x="month",
    y="num_journeys",
    color="operator",
    template="simple_white",
    labels=labels_map,
    title="Nombre de journeys incités par opérateur"
    f"<br><sub>Uniquement les trajets avec O/D à moins de {MAX_DISTANCE / 1000:.0f} km d'une gare</sub>",
)
fig_journeys_by_operator_filtered.update_yaxes(showgrid=True)
fig_journeys_by_operator_filtered.show()


fig_journeys_by_operator_filtered.write_html(
    "outputs_idfm/fig_journeys_par_operateur_mois_filtered.html"
)
fig_journeys_by_operator_filtered.write_image(
    "outputs_idfm/fig_journeys_par_operateur_mois_filtered.svg", width=1280, height=720
)

## Par opérateurs


In [ ]:
px.line(
    (
        df_journeys_raw_with_nearest_stations_full.filter(
            incentived_trip_filter_expr & pl.col("is_near_station")
        )
        .group_by(["operator_id", pl.col("start_datetime").dt.truncate("1w")])
        .agg(pl.col("_id").n_unique().alias("num_journeys"))
        .join(
            df_operators,
            left_on="operator_id",
            right_on="_id",
            validate="m:1",
            suffix="_operators",
        )
        .sort(["start_datetime", "operator_id"])
    ),
    title="Nombre de journeys par opérateur"
    f"<br><sub>Uniquement les trajets avec O/D à moins de {MAX_DISTANCE / 1000:.0f} km d'une gare</sub>",
    x="start_datetime",
    y="num_journeys",
    color="name",
    labels=labels_map,
    template="simple_white",
)

# Comparaisons entre les trajets proches et éloignés d'une gare


In [ ]:
colors_map = {
    "Trajets éloignés d'une gare": "#ffae2b",
    "Trajets proches d'une gare": "#2B7CFF",
}
category_order = ["Trajets proches d'une gare", "Trajets éloignés d'une gare"]

## Distance


In [ ]:
fig_distance_by_month_near_vs_far = px.box(
    df_journeys_raw_with_nearest_stations_full.with_columns(

 (pl.col("distance") / 1000).alias("distance_km"), pl.col("start_datetime").dt.strftime("%Y-%m").alias("year_month")

 ).sort("is_near_station_fmt"),
    x="year_month",
    y="distance_km",
    color="is_near_station_fmt",
    color_discrete_map=colors_map,
    category_orders={"is_near_station_fmt": category_order},
    template="simple_white",
    labels=labels_map,
    title="Distribution de la distance par trajet par mois - "
    f'<br><sub>Les trajets dits "proches" sont ceux avec O/D à moins de {MAX_DISTANCE / 1000:.0f} km d\'une gare<sub>',
)

fig_distance_by_month_near_vs_far.update_yaxes(showgrid=True)

fig_distance_by_month_near_vs_far.update_traces(marker=dict(size=0, opacity=0))
fig_distance_by_month_near_vs_far.update_yaxes(
    range=[0, 80]
)
# fig_distance_by_month_near_vs_far.show()

fig_distance_by_month_near_vs_far.write_html(
    OUTPUT_PATH / "fig_distance_by_month_near_vs_far.html"
)
fig_distance_by_month_near_vs_far.write_image(
    OUTPUT_PATH / "fig_distance_by_month_near_vs_far.svg", width=1280, height=720
)

Les trajets qui débutent et finissent proches d'une gare sont en moyennes plus courts que les autres trajets. Cela pourrait par exemple traduire un déplacement hybride (voiture + transports en communs). S'ajoute à cela le fait que les gares soient des points de rencontre. A priori, la plupart des trajets se dirigent vers Paris - les points d'arrivées sont peut etre des gares qui connectent rapidement au métro/permettent de se rendre rapidement dans Paris. Les gares sont peut-être à proximité de parking relais.
TODO: vérifier les hypothèses pertinentes, trouver d'autres hypothèses.


## Par opérateurs


In [ ]:
fig_journey_by_operateur_by_month_near_vs_far = px.line(
    (
        df_journeys_raw_with_nearest_stations_full.filter(incentived_trip_filter_expr)
        .group_by(
            [
                "operator_id",
                pl.col("start_datetime").dt.truncate("1w"),
                "is_near_station_fmt",
            ]
        )
        .agg(pl.col("_id").n_unique().alias("num_journeys"))
        .with_columns(
            (
                pl.col("num_journeys")
                / pl.col("num_journeys")
                .sum()
                .over(["start_datetime", "is_near_station_fmt"])
                * 100
            ).alias("percentage_journeys")
        )
        .join(
            df_operators,
            left_on="operator_id",
            right_on="_id",
            validate="m:1",
            suffix="_operators",
        )
        .sort(["start_datetime", "operator_id"])
    ),
    title=f"Proportion de journeys par opérateur par catégorie de distance à une gare <br><sub>Les trajets dits proches sont ceux avec O/D à moins de {MAX_DISTANCE / 1000:.0f} km d'une gare</sub>",
    x="start_datetime",
    y="percentage_journeys",
    color="name",
    line_dash="is_near_station_fmt",
    labels=labels_map,
    template="simple_white",
)

fig_journey_by_operateur_by_month_near_vs_far.show()

fig_journey_by_operateur_by_month_near_vs_far.write_html(
    OUTPUT_PATH / "fig_journey_by_operateur_by_month_near_vs_far.html"
)

fig_journey_by_operateur_by_month_near_vs_far.write_image(
    OUTPUT_PATH / "fig_journey_by_operateur_by_month_near_vs_far.svg", width=1280, height=720
)

Les répartitions par opérateurs sont les mêmes que le trajet soit proche d\'une gare ou non. Certains opérateurs sont utilisés uniquement pour les trajets éloignés des gares - surement du à un faible échantillon


## Prix, revenus et incitations


### Trajets incités


In [ ]:
df_stats_by_week_near = (
    df_journeys_raw_with_nearest_stations_full
    .filter(incentived_trip_filter_expr & pl.col("is_near_station"))
    .group_by([pl.col("start_datetime").dt.truncate("1w").alias("week")])
    .agg(agg_expressions)
    .sort("week")
)

df_stats_by_week_far = (
    df_journeys_raw_with_nearest_stations_full
    .filter(incentived_trip_filter_expr & ~pl.col("is_near_station"))
    .group_by([pl.col("start_datetime").dt.truncate("1w").alias("week")])
    .agg(agg_expressions)
    .sort("week")
)

df_stats_by_week_near_far = (
    df_journeys_raw_with_nearest_stations_full
    .filter(incentived_trip_filter_expr)
    .group_by([pl.col("start_datetime").dt.truncate("1w").alias("week"), "is_near_station_fmt"])
    .agg(agg_expressions)
    .sort("week")
)

#### Proche d'une gare


In [ ]:
def create_scatter_fig_prices_near_vs_far(
    df: pl.DataFrame,
    stats_cols: list[str],
    x_col: str,
    title: str,
    labels_map: dict[str, str],
    x_title: str = "Montant (euros)",
    min_y: float = 0,
    max_y: float = None
) -> go.Figure:  
    
    if not max_y:
        max_y = df.select(stats_cols).max().max_horizontal().item()
    
    df = df.unpivot(
        index=[x_col, "is_near_station_fmt"],
        on=stats_cols,
        variable_name="metric",
        value_name="value"
    )
    
    df = df.to_pandas()
    
    fig = px.line(
        df,
        x=x_col,  
        y="value",  
        color="metric",  
        line_dash="is_near_station_fmt",
        labels=labels_map,
        template="simple_white",
    )
    for trace in fig.data:
        original_name = trace.name.split(",")[0]
        if original_name in labels_map:
            trace.name = trace.name.replace(original_name, labels_map[original_name])
    
    fig.update_layout(
        title=title,

    )
    
    fig.update_yaxes(
        range=[min_y, max_y * 1.2],
        title=x_title,
        showgrid=True,
        gridwidth=2,
        ticksuffix="€",
    )
    
    fig.update_xaxes(title="Mois" if x_col == "month" else "Semaine")
    
    return fig

In [ ]:
fig_prices_by_week_near_vs_far = create_scatter_fig_prices_near_vs_far(
    df_stats_by_week_near_far.sort("is_near_station_fmt"),
    [
        "incentive_amount_avg",
        "passenger_contribution_incentived_trips_avg",
        "driver_revenue_incentived_trips_avg",
    ],
    "week",
    (
        "Montants moyens par trajet des incitations par catégorie de distance à une gare,"
        "<br>contributions passagers et revenus conducteurs - Trajets incités"
    ),
    {
        **labels_map,
        "passenger_contribution_incentived_trips_avg": "Contribution moyenne passager",
        "driver_revenue_incentived_trips_avg": "Revenu moyen conducteur",
    },
    max_y=3.1
)
fig_prices_by_week_near_vs_far.show()


fig_prices_by_week_near_vs_far.write_html(OUTPUT_PATH / "fig_prices_by_week_near_vs_far.html")
fig_prices_by_week_near_vs_far.write_image(
    OUTPUT_PATH / "fig_prices_by_week_near_vs_far.svg", width=1280, height=720
)

#### Eloignés d'une gare


In [ ]:
def create_scatter_fig_prices(
    df: pl.DataFrame,
    stats_cols: list[str],
    x_col: str,
    title: str,
    labels_map: dict[str, str],
    x_title: str = "Montant (euros)",
    min_y: float = 0,
    max_y: float = None
) -> go.Figure:
    traces = []
    for name in stats_cols:
        trace = go.Scatter(
            x=df[x_col],
            y=df[name],
            name=labels_map.get(name, name),
            mode="lines+markers",
            marker_size=4,
        )
        traces.append(trace)
    fig = go.Figure(traces)
    fig.update_layout(
        template="simple_white",
        title=title,
        legend_orientation="h",
        legend_y=0.7,
        legend_yref="container",
    )

    if not max_y:
        max_y = df.select(stats_cols).max().max_horizontal().item()

    fig.update_yaxes(
        range=[min_y, max_y * 1.2],
        title=x_title,
        showgrid=True,
        gridwidth=2,
        ticksuffix="€",
    )
    fig.update_xaxes(title="Mois" if x_col == "month" else "Semaine")

    return fig

In [ ]:
fig_prices_by_week_far = create_scatter_fig_prices(
    df_stats_by_week_far,
    [
        "incentive_amount_avg",
        "passenger_contribution_incentived_trips_avg",
        "driver_revenue_incentived_trips_avg",
    ],
    "week",
    (
        "Montants moyens par trajet des incitations pour les trajets <b>éloignés d'une gare</b>,"
        "<br>contributions passagers et revenus conducteurs - Trajets incités"
    ),
    {
        **labels_map,
        "passenger_contribution_incentived_trips_avg": "Contribution moyenne passager",
        "driver_revenue_incentived_trips_avg": "Revenu moyen conducteur",
    },
    max_y=3.1
)
fig_prices_by_week_far.show()


fig_prices_by_week_far.write_html(OUTPUT_PATH / "fig_prix_par_semaine_far.html")
fig_prices_by_week_far.write_image(
    OUTPUT_PATH / "fig_prix_par_semaine_far.svg", width=1280, height=720
)

#### Intra


##### Proches d'une gare


In [ ]:
fig_prices_by_week_intra_near = create_scatter_fig_prices(
    df_stats_by_week_near,
    [
        "incentive_amount_intra_avg",
        "passenger_contribution_intra_avg",
        "driver_revenue_intra_avg",
    ],
    "week",
    (
        "Montants moyens par trajet <b>intra proche d'une gare</b>  des incitations,"
        "<br>contributions passagers et revenus conducteurs"
    ),
    labels_map,
    max_y=3
)
fig_prices_by_week_intra_near.show()


fig_prices_by_week_intra_near.write_html(OUTPUT_PATH / "fig_prix_intra_par_semaine_near.html")
fig_prices_by_week_intra_near.write_image(
    OUTPUT_PATH / "fig_prix_intra_par_semaine_near.svg", width=1280, height=720
)

##### Eloignés d'une gare


In [ ]:
fig_prices_by_week_intra_far = create_scatter_fig_prices(
    df_stats_by_week_far,
    [
        "incentive_amount_intra_avg",
        "passenger_contribution_intra_avg",
        "driver_revenue_intra_avg",
    ],
    "week",
    (
        "Montants moyens par trajet <b>intra éloigné d'une gare</b> des incitations,"
        "<br>contributions passagers et revenus conducteurs"
    ),
    labels_map,
    max_y=3
)
fig_prices_by_week_intra_far.show()


fig_prices_by_week_intra_far.write_html(OUTPUT_PATH / "fig_prix_intra_par_semaine_far.html")
fig_prices_by_week_intra_far.write_image(
    OUTPUT_PATH / "fig_prix_intra_par_semaine_far.svg", width=1280, height=720
)

#### Inter


##### Proches d'une gare


In [ ]:
fig_prices_by_week_inter_near = create_scatter_fig_prices(
    df_stats_by_week_near,
    [
        "incentive_amount_inter_avg",
        "passenger_contribution_inter_avg",
        "driver_revenue_inter_avg",
    ],
    "week",
    (
        "Montants moyens par trajet <b>inter proche d'une gare</b>  des incitations,"
        "<br>contributions passagers et revenus conducteurs"
    ),
    labels_map,
    max_y=6
)
fig_prices_by_week_inter_near.show()


fig_prices_by_week_inter_near.write_html(OUTPUT_PATH / "fig_prix_inter_par_semaine_near.html")
fig_prices_by_week_inter_near.write_image(
    OUTPUT_PATH / "fig_prix_inter_par_semaine_near.svg", width=1280, height=720
)

##### Eloignés d'une gare


In [ ]:
fig_prices_by_week_inter_far = create_scatter_fig_prices(
    df_stats_by_week_far,
    [
        "incentive_amount_inter_avg",
        "passenger_contribution_inter_avg",
        "driver_revenue_inter_avg",
    ],
    "week",
    (
        "Montants moyens par trajet <b>inter éloigné d'une gare</b> des incitations,"
        "<br>contributions passagers et revenus conducteurs"
    ),
    labels_map,
    max_y=6
)
fig_prices_by_week_inter_far.show()


fig_prices_by_week_inter_far.write_html(OUTPUT_PATH / "fig_prix_inter_par_semaine_far.html")
fig_prices_by_week_inter_far.write_image(
    OUTPUT_PATH / "fig_prix_inter_par_semaine_far.svg", width=1280, height=720
)

## Nombre de passagers


In [ ]:
df_stats_with_nearest_stations_full = (
    df_journeys_raw_with_nearest_stations_full.filter(incentived_trip_filter_expr)
    .group_by(pl.col("is_near_station"))
    .agg(agg_expressions)
)
df_stats_with_nearest_stations_full.select(
    pl.col("number_of_unique_passenger").sum().alias("Nombre de passagers"),
    pl.col("number_of_unique_passenger")
    .filter(
        pl.col("is_near_station"),
    )
    .alias("Nombre de passagers effectuants des trajets proches de gare"),
    pl.col("number_of_unique_passenger")
    .filter(
        pl.col("is_near_station").not_(),
    )
    .alias("Nombre de passagers effectuants des trajets éloignés de gare"),
)

In [ ]:
print(df_journeys_raw_with_nearest_stations_full.columns)

In [ ]:


fig_num_passenger_by_week = px.line(
    (
        df_journeys_raw_with_nearest_stations_full.filter(
            incentived_trip_filter_expr
        )
        .group_by([pl.col("start_datetime").dt.truncate("1w"), "is_near_station_fmt"])
        .agg(pl.col("passenger_identity_key").n_unique().alias("num_passenger"))
        .sort(["start_datetime", "is_near_station_fmt"])
    ),
    title=f"Nombre de passager par catégorie de distance à une gare <br><sub>Les trajets dits proches sont ceux avec O/D à moins de {MAX_DISTANCE / 1000:.0f} km d\'une gare</sub>",
    x="start_datetime",
    y="num_passenger",
    color="is_near_station_fmt",
    color_discrete_map=colors_map,
    category_orders={"is_near_station_fmt": category_order},
    labels=labels_map,
    template="simple_white",
)

fig_num_passenger_by_week.show()


fig_num_passenger_by_week.write_html(OUTPUT_PATH / "fig_num_passenger_par_semaine.html")
fig_num_passenger_by_week.write_image(
    OUTPUT_PATH / "fig_num_passenger_par_semaine.svg", width=1280, height=720
)

## Conducteurs


### Nombre de conducteurs

In [ ]:
df_stats_with_nearest_stations_full.select(
        pl.col("number_of_unique_driver").sum().alias("Nombre de conducteurs"),
        pl.col("number_of_unique_driver")
        .filter(
            pl.col("is_near_station"),
            )
        .alias(
        "Nombre de conducteurs effectuants des trajets proches de gare"
        ), 
        pl.col("number_of_unique_driver").filter(pl.col("is_near_station").not_(),
        ).alias("Nombre de conducteurs effectuants des trajets éloignés de gare"),
    )

### Acquisition


In [ ]:
fig_new_drivers_count_by_week = px.bar(
    df_journeys_raw_with_nearest_stations_full.filter(
        pl.col("first_trip_datetime") >= datetime(2024, 9, 1, tzinfo=ZoneInfo("GMT"))
    )
    .group_by(
        [
            pl.col("first_trip_datetime").dt.truncate("1w").alias("week"),
            "is_near_station_fmt",
        ]
    )
    .agg(pl.len())
    .sort(pl.col("week")),
    x="week",
    y="len",
    color="is_near_station_fmt",
    color_discrete_map=colors_map,
    category_orders={"is_near_station_fmt": category_order},
    labels={**labels_map, "len": "Nombre de nouveaux conducteurs"},
    template="simple_white",
    title="Evolution de l'acquisition des conducteurs",
    barmode="group",
)
fig_new_drivers_count_by_week.show()

fig_new_drivers_count_by_week.write_html(
    OUTPUT_PATH / "fig_conducteurs_par_semaine.html"
)
fig_new_drivers_count_by_week.write_image(
    OUTPUT_PATH / "fig_conducteurs_par_semaine.svg", width=1280, height=720
)

## Nombre de trajets


In [ ]:
df_journeys_raw_with_nearest_stations_full.filter(
    pl.col("first_trip_datetime") >= datetime(2024, 9, 1, tzinfo=ZoneInfo("GMT")),
    pl.col("first_trip_datetime") <= datetime.now(ZoneInfo("GMT")) - timedelta(days=14),
).group_by(
    [
        "is_near_station_fmt",
        "driver_identity_key",
        pl.col("start_datetime").dt.truncate("1w"),
    ]
).agg(
    pl.len().alias("num_journeys"),
    pl.concat_str(pl.col("operator_id"), pl.lit("-"), pl.col("operator_trip_id"))
    .n_unique()
    .alias("num_trips"),
).group_by(["is_near_station_fmt", "start_datetime"]).agg(
    pl.col("num_journeys").mean().alias("Nombre moyen de journeys par semaine"),
    pl.col("num_trips").mean().alias("Nombre moyen de trips par semaine"),
).group_by(["is_near_station_fmt"]).agg(
    pl.col("Nombre moyen de journeys par semaine").mean(),
    pl.col("Nombre moyen de trips par semaine").mean(),
)

In [ ]:
df_journeys_raw_with_nearest_stations_full.filter(
    pl.col("first_trip_datetime") >= datetime(2024, 9, 1, tzinfo=ZoneInfo("GMT")),
    pl.col("first_trip_datetime") <= datetime.now(ZoneInfo("GMT")) - timedelta(days=30),
    pl.col("start_datetime") <= pl.col("first_trip_datetime") + pl.duration(days=30),
).group_by(["is_near_station_fmt", "driver_identity_key"]).agg(
    pl.len().alias("num_journeys"),
    pl.concat_str(pl.col("operator_id"), pl.lit("-"), pl.col("operator_trip_id"))
    .n_unique()
    .alias("num_trips"),
).group_by("is_near_station_fmt").agg(
    pl.col("num_journeys").mean().alias("Nombre moyen de journeys sur 30 jours"),
    pl.col("num_trips").mean().alias("Nombre moyen de trips sur 30 jours"),
)

In [ ]:
def create_num_drivers_by_num_trips_hist_fig(
    df: pl.DataFrame, step_size: int, max_step: int
) -> go.Figure:
    breaks = range(1, max_step + 1, step_size)

    station_distance_categories = (
        df_journeys_raw_with_nearest_stations_full.select("is_near_station_fmt")
        .unique()
        .to_series()
        .to_list()
    )

    # Création du DataFrame de toutes les combinaisons possibles
    combinations = pl.DataFrame(
        product(station_distance_categories, breaks),
        schema=["is_near_station_fmt", "breaks_raw"],
    ).with_columns(
        pl.col("breaks_raw")
        .cut(breaks, include_breaks=True, left_closed=True)
        .struct.unnest()
    )
    data_agg = (
        df.filter(
            pl.col("first_trip_datetime")
            >= datetime(2024, 9, 1, tzinfo=ZoneInfo("GMT")),
            pl.col("first_trip_datetime")
            <= datetime.now(ZoneInfo("GMT")) - timedelta(days=30),
            pl.col("start_datetime")
            <= pl.col("first_trip_datetime") + pl.duration(days=30),
        )
        .group_by(["is_near_station_fmt", "driver_identity_key"])
        .agg(
            pl.len().alias("num_journeys"),
            pl.concat_str(
                pl.col("operator_id"), pl.lit("-"), pl.col("operator_trip_id")
            )
            .n_unique()
            .alias("num_trips"),
        )
        .with_columns(
            pl.col("num_trips").cut(
                breaks=breaks, left_closed=True, include_breaks=True
            )
        )
        .group_by(["is_near_station_fmt", "num_trips"])
        .agg(pl.col("driver_identity_key").n_unique().alias("num_drivers"))
        .with_columns(pl.col("num_trips").struct.unnest())
    )

    data_complete = (
        combinations.join(
            data_agg,
            on=["is_near_station_fmt", "breakpoint"],
            how="left",
        )
        .with_columns(pl.col("num_drivers").fill_null(0))
        .with_columns(
            (
                100
                * pl.col("num_drivers")
                / pl.col("num_drivers").sum().over("is_near_station_fmt")
            )
            .round(2)
            .alias("share_drivers")
        )
        .sort(["breakpoint"])
    )

    fig = px.bar(
        data_complete,
        x=data_complete["category"],
        y=data_complete["share_drivers"],
        color="is_near_station_fmt",
        color_discrete_map=colors_map,
        # category_orders={"is_near_station_fmt": category_order},
        barmode="group",
        template="simple_white",
        title="Distribution du nombre de trajets conducteurs effectués sur 30 jours pour chaque catégorie de distance à une gare",
    )

    fig.update_xaxes(title="Nombre de trajets conducteurs")
    fig.update_yaxes(title="% des conducteurs")

    return fig, data_complete


fig_drivers_by_trip_numbers_hist, data_complete = (
    create_num_drivers_by_num_trips_hist_fig(
        df_journeys_raw_with_nearest_stations_full, step_size=3, max_step=30
    )
)
fig_drivers_by_trip_numbers_hist.show()

fig_drivers_by_trip_numbers_hist.write_html(
    OUTPUT_PATH / "fig_histo_trajets_conducteurs_near_far.html"
)
fig_drivers_by_trip_numbers_hist.write_image(
    OUTPUT_PATH / "fig_histo_trajets_conducteurs_near_far.svg", width=1280, height=720
)

### Cohérence comportementale conducteur

In [ ]:
def create_behavioral_consistency_chart(
    df: pl.DataFrame,
    entity_type: str,
    category_col: str,  
    category_true_label: str,  
    category_false_label: str,  
    start_date: datetime = DEFAULT_START_DATE,
    period_days: int = 30,
    labels_map: dict = None,
    prop_breaks: list = None,
    color_discrete_map: dict = None,
) -> tuple[go.Figure, pl.DataFrame]:
    """
    Analyse la cohérence comportementale des entités sur une colonne binaire.
    
    Args:
        df: DataFrame Polars
        entity_type: "driver" ou "passenger"
        category_col: Colonne binaire à analyser (ex: "is_near_station")
        category_true_label: Label pour la catégorie True (ex: "proches")
        category_false_label: Label pour la catégorie False (ex: "éloignés")
        start_date: Date de début du filtre
        period_days: Nombre de jours de la période d'analyse
        labels_map: Dictionnaire de mapping des labels
        prop_breaks: Breaks pour les proportions (par défaut [0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0])
        color_discrete_map: Mapping des couleurs pour les catégories
    
    Returns:
        Tuple (Figure plotly, DataFrame des données)
    """
    if entity_type not in ENTITY_CONFIGS:
        raise ValueError(f"entity_type doit être dans {list(ENTITY_CONFIGS.keys())}")
    
    config = ENTITY_CONFIGS[entity_type]
    identity_col = config["identity_col"]
    first_trip_col = config["first_trip_col"]
    entity_label = config["label_plural"]
    entity_label_singular = config["label_singular"]
    
    if prop_breaks is None:
        prop_breaks = [0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0]
    
    if labels_map is None:
        labels_map = {}
    
    majority_true_category = f"{entity_label_singular.capitalize()} effectuant des trajets majoritairement {category_true_label}"
    majority_false_category = f"{entity_label_singular.capitalize()} effectuant des trajets majoritairement {category_false_label}"
    
    if color_discrete_map is None:
        color_discrete_map = {
            majority_true_category: "#2B7CFF",
            majority_false_category: "#ffae2b"
        }
    
    entity_profiles = (
        df.filter(
            pl.col(first_trip_col) >= start_date,
            pl.col(first_trip_col) <= datetime.now(ZoneInfo("GMT")) - timedelta(days=period_days),
            pl.col("start_datetime") <= pl.col(first_trip_col) + pl.duration(days=period_days),
        )
        .group_by(identity_col)
        .agg([
            pl.col(category_col).count().alias("total_trips"),
            pl.col(category_col).sum().alias(f"trips_{category_true_label}"),
            (pl.col(category_col).sum() / pl.col(category_col).count()).alias(f"prop_{category_true_label}")
        ])
        .with_columns([
            pl.when(pl.col(f"prop_{category_true_label}") > 0.5)
            .then(pl.lit(majority_true_category))
            .otherwise(pl.lit(majority_false_category))
            .alias(f"{entity_type}_category")
        ])
        .with_columns(
            pl.col(f"prop_{category_true_label}")
            .cut(prop_breaks, include_breaks=True, left_closed=True)
            .struct.unnest()
        )
    )
    
    entity_categories = [majority_true_category, majority_false_category]
    combinations = pl.DataFrame(
        list(product(entity_categories, prop_breaks)), 
        schema=[f"{entity_type}_category", "breaks_raw"]
    ).with_columns(
        pl.col("breaks_raw")
        .cut(prop_breaks, include_breaks=True, left_closed=True)
        .struct.unnest()
    )
    
    data_agg = (
        entity_profiles
        .group_by([f"{entity_type}_category", "breakpoint"])
        .agg([
            pl.len().alias(f"num_{entity_type}s"),
            pl.col(f"prop_{category_true_label}").mean().alias("avg_prop_in_bucket")
        ])
    )
    
    data_complete = (
        combinations
        .join(data_agg, on=[f"{entity_type}_category", "breakpoint"], how="left")
        .with_columns([
            pl.col(f"num_{entity_type}s").fill_null(0),
            pl.col("avg_prop_in_bucket").fill_null(0)
        ])
        .with_columns(
            (100 * pl.col(f"num_{entity_type}s") / pl.col(f"num_{entity_type}s").sum().over(f"{entity_type}_category"))
            .round(2)
            .alias(f"share_{entity_type}s")
        )
        .sort(["breakpoint", f"{entity_type}_category"])
    )
    
    data_pandas = data_complete.to_pandas()
    
    fig = px.bar(
        data_pandas,
        x="category",
        y=f"share_{entity_type}s",
        color=f"{entity_type}_category",
        color_discrete_map=color_discrete_map,
        category_orders={
            f"{entity_type}_category": [majority_true_category, majority_false_category]
        },
        barmode="group",
        template="simple_white",
        title=f"Distribution de la cohérence comportementale des {entity_label}<br><sub>% de trips {category_true_label} vs profil majoritaire du {entity_label_singular}</sub>",
        labels={
            **labels_map,
            "category": f"Proportion de trips {category_true_label}",
            f"share_{entity_type}s": f"% des {entity_label}",
            f"{entity_type}_category": "Profil majoritaire"
        }
    )
    
    fig.update_xaxes(title=f"Proportion de trips {category_true_label}")
    fig.update_yaxes(title=f"% des {entity_label}")
    fig.update_layout(
        legend_orientation="h",
        legend_y=1.02,
        legend_yref="container"
    )
    
    return fig, data_complete


def get_consistency_summary(
    data_complete: pl.DataFrame,
    entity_type: str,
) -> pl.DataFrame:
    """
    Calcule les statistiques de cohérence à partir des données complètes.
    
    Args:
        data_complete: DataFrame résultat de create_behavioral_consistency_chart
        entity_type: "driver" ou "passenger"
    
    Returns:
        DataFrame avec les stats de cohérence par catégorie
    """
    consistency_summary = (
        data_complete
        .filter(pl.col(f"num_{entity_type}s") > 0)
        .group_by(f"{entity_type}_category")
        .agg([
            (pl.col(f"num_{entity_type}s") * pl.col("avg_prop_in_bucket")).sum().alias("weighted_sum"),
            pl.col(f"num_{entity_type}s").sum().alias(f"total_{entity_type}s")
        ])
        .with_columns(
            (pl.col("weighted_sum") / pl.col(f"total_{entity_type}s")).round(3).alias("avg_consistency")
        )
    )
    return consistency_summary

In [ ]:
fig_driver_consistency, data_driver_consistency = create_behavioral_consistency_chart(
    df=df_journeys_raw_with_nearest_stations_full,
    entity_type="driver",
    category_col="is_near_station",
    category_true_label="proches",
    category_false_label="éloignés",
    labels_map=labels_map,
)

fig_driver_consistency.show()
fig_driver_consistency.write_html(OUTPUT_PATH / "fig_driver_consistency.html")
fig_driver_consistency.write_image(
    OUTPUT_PATH / "fig_driver_consistency.svg", width=1280, height=720
)

consistency_summary = get_consistency_summary(data_driver_consistency, "driver")
print(consistency_summary.to_pandas())

### Types de trajets


In [ ]:
df_journeys_raw_with_nearest_stations_full.filter(
    pl.col("first_trip_datetime") >= datetime(2024, 9, 1, tzinfo=ZoneInfo("GMT")),
    pl.col("first_trip_datetime") <= datetime.now(ZoneInfo("GMT")) - timedelta(days=30),
    pl.col("start_datetime") <= pl.col("first_trip_datetime") + pl.duration(days=30),
).group_by(["is_near_station_fmt", "driver_identity_key"]).agg(
    (
        (
            pl.concat_str(
                pl.col("operator_id"), pl.lit("-"), pl.col("operator_trip_id")
            )
            .filter(pl.col("is_fully_inside_campaign_area"))
            .n_unique()
        )
        >= (
            pl.concat_str(
                pl.col("operator_id"), pl.lit("-"), pl.col("operator_trip_id")
            )
            .filter(pl.col("is_fully_inside_campaign_area").not_())
            .n_unique()
        )
    ).alias("is_intra_driver")
).group_by(["is_near_station_fmt"]).agg(
    (100 * pl.col("is_intra_driver").sum() / pl.len()).alias(
        "% des conducteurs avec une majorité de journeys intra"
    )
)

In [ ]:
df_journeys_trips_count_by_trip_type = (
    df_journeys_raw_with_nearest_stations_full.filter(
        pl.col("first_trip_datetime") >= datetime(2024, 9, 1, tzinfo=ZoneInfo("GMT")),
        pl.col("first_trip_datetime")
        <= datetime.now(ZoneInfo("GMT")) - timedelta(days=30),
        pl.col("start_datetime")
        <= pl.col("first_trip_datetime") + pl.duration(days=30),
    )
    .group_by(["is_near_station_fmt", "is_fully_inside_campaign_area", "driver_identity_key"])
    .agg(
        pl.len().alias("num_journeys"),
        pl.concat_str(pl.col("operator_id"), pl.lit("-"), pl.col("operator_trip_id"))
        .n_unique()
        .alias("num_trips"),
    )
    .group_by(["is_near_station_fmt", "is_fully_inside_campaign_area"])
    .agg(
        pl.col("num_journeys").mean().alias("Nombre moyen de journeys sur 30 jours"),
        pl.col("num_trips").mean().alias("Nombre moyen de trips sur 30 jours"),
    )
    
)
df_journeys_trips_count_by_trip_type

In [ ]:
fig_journeys_count_by_driver_type_station_proximity = px.bar(
    df_journeys_trips_count_by_trip_type.with_columns(
        pl.when(pl.col("is_fully_inside_campaign_area"))
        .then(pl.lit("Trajet intra"))
        .otherwise(pl.lit("Trajet inter"))
        .alias("driver_type")
    ),
    x="is_near_station_fmt",
    y="Nombre moyen de journeys sur 30 jours",
    color="is_fully_inside_campaign_area",
    text="Nombre moyen de journeys sur 30 jours",
    text_auto=".1f",
    template="simple_white",
    barmode="group",
    labels=labels_map,
    title="Nombre de journeys par type de conducteur et distance d'une gare",
)
fig_journeys_count_by_driver_type_station_proximity.update_layout(legend_title=None)
fig_journeys_count_by_driver_type_station_proximity.show()
fig_journeys_count_by_driver_type_station_proximity.write_html(
    OUTPUT_PATH / "fig_journeys_par_type_conducteur_et_distance_gare.html"
)
fig_journeys_count_by_driver_type_station_proximity.write_image(
    OUTPUT_PATH / "fig_journeys_par_type_conducteur_et_distance_gare.svg", width=1280, height=720
)

In [ ]:
fig_trips_count_by_driver_type_station_proximity = px.bar(
    df_journeys_trips_count_by_trip_type.with_columns(
        pl.when(pl.col("is_fully_inside_campaign_area"))
        .then(pl.lit("Trajet intra"))
        .otherwise(pl.lit("Trajet inter"))
        .alias("driver_type")
    ),
    x="is_near_station_fmt",
    y="Nombre moyen de trips sur 30 jours",
    color="driver_type",
    text="Nombre moyen de trips sur 30 jours",
    text_auto=".1f",
    template="simple_white",
    barmode="group",
    labels=labels_map,
    title="Nombre de trips par type de conducteur et distance d'une gare",
)

fig_trips_count_by_driver_type_station_proximity.update_layout(legend_title=None)
fig_trips_count_by_driver_type_station_proximity.show()
fig_trips_count_by_driver_type_station_proximity.write_html(
    OUTPUT_PATH / "fig_trips_par_type_conducteur_et_distance_gare.html"
)
fig_trips_count_by_driver_type_station_proximity.write_image(
    OUTPUT_PATH/"fig_trips_par_type_conducteur_et_distance_gare.svg", width=1280, height=720
)

In [ ]:
df_passenger_mean_by_distance_cat = (
    df_journeys_raw_with_nearest_stations_full.filter(
        pl.col("first_trip_datetime") >= datetime(2024, 9, 1, tzinfo=ZoneInfo("GMT")),
        pl.col("first_trip_datetime")
        <= datetime.now(ZoneInfo("GMT")) - timedelta(days=30),
        pl.col("start_datetime")
        <= pl.col("first_trip_datetime") + pl.duration(days=30),
    )
    .group_by(
        [
            pl.concat_str(
                pl.col("operator_id"), pl.lit("-"), pl.col("operator_trip_id")
            ).alias("trip_id"),
        ]
    )
    .agg(
        pl.col("is_fully_inside_campaign_area").max(),
        pl.col("passenger_seats").sum(),
        pl.col("is_near_station_fmt").mode().first().alias("is_near_station_fmt"),
        pl.col("driver_identity_key").max(),
    )
    .group_by(["is_near_station_fmt", "driver_identity_key", "is_fully_inside_campaign_area"])
    .agg(
        (
            (
                pl.col("trip_id")
                .filter(pl.col("is_fully_inside_campaign_area"))
                .n_unique()
            )
            >= (
                pl.col("trip_id")
                .filter(pl.col("is_fully_inside_campaign_area").not_())
                .n_unique()
            )
        ).alias("is_intra_driver"),
        pl.len().alias("num_journeys"),
        pl.col("passenger_seats").mean(),
    )
    .group_by(["is_near_station_fmt", "is_fully_inside_campaign_area"])
    .agg(
        pl.col("passenger_seats").mean().alias("Nombre moyen de passagers"),
    )
)
df_passenger_mean_by_distance_cat

In [ ]:
fig_passengers_count_by_driver_type_distance_gare = px.bar(
    df_passenger_mean_by_distance_cat.with_columns(
        pl.when(pl.col("is_fully_inside_campaign_area"))
        .then(pl.lit("Trajet intra"))
        .otherwise(pl.lit("Trajet inter"))
        .alias("driver_type")
    ),
    x="is_near_station_fmt",
    y="Nombre moyen de passagers",
    color="driver_type",
    text="Nombre moyen de passagers",
    text_auto=".2f",
    template="simple_white",
    barmode="group",
    labels=labels_map,
    title="Nombre moyen de passagers par type de conducteur et campagne",
)

fig_passengers_count_by_driver_type_distance_gare.update_layout(legend_title=None)
fig_passengers_count_by_driver_type_distance_gare.show()
fig_passengers_count_by_driver_type_distance_gare.write_html(
    OUTPUT_PATH / "fig_passagers_type_conducteur_et_distance_gare.html"
)
fig_passengers_count_by_driver_type_distance_gare.write_image(
    OUTPUT_PATH / "fig_passagers_par_type_conducteur_et_distance_gare.svg", width=1280, height=720
)

In [ ]:
df_passenger_mean_by_distance_cat_post_cee = (
    df_journeys_raw_with_nearest_stations_full.filter(
        pl.col("first_trip_datetime") >= datetime(2025, 3, 1, tzinfo=ZoneInfo("GMT")),
        pl.col("first_trip_datetime")
        <= datetime.now(ZoneInfo("GMT")) - timedelta(days=30),
        pl.col("start_datetime")
        <= pl.col("first_trip_datetime") + pl.duration(days=30),
    )
    .group_by(
        [
            pl.concat_str(
                pl.col("operator_id"), pl.lit("-"), pl.col("operator_trip_id")
            ).alias("trip_id"),
        ]
    )
    .agg(
        pl.col("is_fully_inside_campaign_area").max(),
        pl.col("passenger_seats").sum(),
        pl.col("is_near_station_fmt").mode().first().alias("is_near_station_fmt"),
        pl.col("driver_identity_key").max(),
    )
    .group_by(["is_near_station_fmt", "driver_identity_key", "is_fully_inside_campaign_area"])
    .agg(
        (
            (
                pl.col("trip_id")
                .filter(pl.col("is_fully_inside_campaign_area"))
                .n_unique()
            )
            >= (
                pl.col("trip_id")
                .filter(pl.col("is_fully_inside_campaign_area").not_())
                .n_unique()
            )
        ).alias("is_intra_driver"),
        pl.len().alias("num_journeys"),
        pl.col("passenger_seats").mean(),
    )
    .group_by(["is_near_station_fmt", "is_fully_inside_campaign_area"])
    .agg(
        pl.col("passenger_seats").mean().alias("Nombre moyen de passagers"),
    )
)
df_passenger_mean_by_distance_cat

In [ ]:
fig_passengers_count_by_driver_type_distance_gare_post_cee = px.bar(
    df_passenger_mean_by_distance_cat_post_cee.with_columns(
        pl.when(pl.col("is_fully_inside_campaign_area"))
        .then(pl.lit("Trajet intra"))
        .otherwise(pl.lit("Trajet inter"))
        .alias("driver_type")
    ),
    x="is_near_station_fmt",
    y="Nombre moyen de passagers",
    color="driver_type",
    text="Nombre moyen de passagers",
    text_auto=".2f",
    template="simple_white",
    barmode="group",
    labels=labels_map,
    title="Nombre moyen de passagers par type de conducteur et campagne",
)

fig_passengers_count_by_driver_type_distance_gare_post_cee.update_layout(legend_title=None)
fig_passengers_count_by_driver_type_distance_gare_post_cee.show()
fig_passengers_count_by_driver_type_distance_gare_post_cee.write_html(
    OUTPUT_PATH / "fig_passagers_type_conducteur_et_distance_gare_post_cee.html"
)
fig_passengers_count_by_driver_type_distance_gare_post_cee.write_image(
    OUTPUT_PATH / "fig_passagers_par_type_conducteur_et_distance_gare_post_cee.svg", width=1280, height=720
)

### Rétention


In [ ]:
def filter_specialized_entities(
    df: pl.DataFrame,
    entity_type: str,
    category_col: str,
    start_date: datetime = DEFAULT_START_DATE,
    period_days: int = 30,
) -> pl.DataFrame:
    """
    Filtre les entités qui font 100% de leurs trajets dans UNE SEULE catégorie.
    
    Args:
        df: DataFrame Polars
        entity_type: "driver" ou "passenger"
        category_col: Colonne de catégorie (ex: "is_near_station_fmt")
        start_date: Date de début du filtre
        period_days: Nombre de jours de la période d'analyse
    
    Returns:
        DataFrame avec [entity_identity_key, entity_category] pour les entités spécialisées
    """
    if entity_type not in ENTITY_CONFIGS:
        raise ValueError(f"entity_type doit être dans {list(ENTITY_CONFIGS.keys())}")
    
    config = ENTITY_CONFIGS[entity_type]
    identity_col = config["identity_col"]
    first_trip_col = config["first_trip_col"]
    
    specialized_entities = (
        df.filter(
            pl.col(first_trip_col) >= start_date,
            pl.col(first_trip_col) <= datetime.now(ZoneInfo("GMT")) - timedelta(days=period_days),
            pl.col("start_datetime") <= pl.col(first_trip_col) + pl.duration(days=period_days),
        )
        .group_by(identity_col)
        .agg([
            pl.col(category_col).n_unique().alias("num_categories"),
            pl.col(category_col).first().alias(f"{entity_type}_category")
        ])
        .filter(pl.col("num_categories") == 1)
        .select([identity_col, f"{entity_type}_category"])
    )
    
    return specialized_entities

In [ ]:
specialized_drivers = filter_specialized_entities(
    df=df_journeys_raw_with_nearest_stations_full,
    entity_type="driver",
    category_col="is_near_station_fmt",
)

df_specialized_drivers = df_journeys_raw_with_nearest_stations_full.join(
    specialized_drivers,
    on="driver_identity_key",
    how="inner"
)

In [ ]:
def create_retention_analysis(
    df: pl.DataFrame,
    entity_type: str,
    start_date: datetime = DEFAULT_START_DATE,
    num_weeks: int = 12,
    group_by_col: str | None = None,
) -> pl.DataFrame:
    """
    Analyse la rétention des entités semaine par semaine avec groupement optionnel.
    
    Args:
        df: DataFrame Polars
        entity_type: "driver" ou "passenger"
        start_date: Date de début du filtre
        num_weeks: Nombre de semaines à analyser après la première course
        group_by_col: Colonne optionnelle pour grouper (ex: "is_near_station_fmt")
    
    Returns:
        DataFrame avec week_number, share_{entity_type}s, et group_by_col si fourni
    """
    if entity_type not in ENTITY_CONFIGS:
        raise ValueError(f"entity_type doit être dans {list(ENTITY_CONFIGS.keys())}")
    
    config = ENTITY_CONFIGS[entity_type]
    identity_col = config["identity_col"]
    first_trip_col = config["first_trip_col"]
    
    initial_group_cols = [identity_col]
    if group_by_col:
        initial_group_cols.append(group_by_col)
    
    partition_cols = [identity_col]
    if group_by_col:
        partition_cols.append(group_by_col)
    
    final_group_cols = ["week_number"]
    if group_by_col:
        final_group_cols.append(group_by_col)
    
    df_retention = (
        df.filter(
            pl.col(first_trip_col) >= start_date,
            pl.col(first_trip_col) <= datetime.now(ZoneInfo("GMT")) - timedelta(weeks=num_weeks),
        )
        .group_by(initial_group_cols)
        .agg(
            pl.col("start_datetime").min(),
            pl.datetime_range(
                pl.col("start_datetime").min().dt.truncate("1w"),
                pl.col("start_datetime").min().dt.truncate("1w") + pl.duration(weeks=num_weeks),
                "1w",
            ).alias("week"),
        )
        .explode("week")
        .join(
            df.filter(
                pl.col("start_datetime") >= start_date,
            ),
            left_on=[identity_col, "week"] + ([group_by_col] if group_by_col else []),
            right_on=[
                identity_col,
                pl.col("start_datetime").dt.truncate("1w"),
            ] + ([group_by_col] if group_by_col else []),
            how="left",
        )
        .group_by([pl.col(identity_col), "week"] + ([group_by_col] if group_by_col else []))
        .agg(
            (pl.col("_id").count() > 0).alias("has_traveled"),
        )
        .with_columns(
            pl.col("week")
            .rank()
            .over(partition_by=partition_cols, order_by="week")
            .alias("week_number")
        )
        .group_by(final_group_cols)
        .agg(
            (100 * pl.col("has_traveled").sum() / pl.col("has_traveled").count()).alias(
                f"{entity_type}s_share"
            )
        )
        .sort(["week_number"] + ([group_by_col] if group_by_col else []))
    )
    
    return df_retention


def plot_retention_analysis(
    df_retention: pl.DataFrame,
    entity_type: str,
    labels_map: dict = None,
    num_weeks: int = 12,
    group_by_col: str | None = None,
    color_discrete_map: dict | None = None,
) -> go.Figure:
    """
    Crée un graphique de rétention à partir du DataFrame d'analyse avec groupement optionnel.
    
    Args:
        df_retention: DataFrame résultat de create_retention_analysis
        entity_type: "driver" ou "passenger"
        labels_map: Dictionnaire de mapping des labels
        num_weeks: Nombre de semaines analysées
        group_by_col: Colonne optionnelle pour grouper (ex: "is_near_station_fmt")
        color_discrete_map: Mapping des couleurs pour le groupement
    
    Returns:
        Figure plotly
    """
    if entity_type not in ENTITY_CONFIGS:
        raise ValueError(f"entity_type doit être dans {list(ENTITY_CONFIGS.keys())}")
    
    config = ENTITY_CONFIGS[entity_type]
    entity_label = config["label_plural"]
    
    if labels_map is None:
        labels_map = {}
    
    chart_labels = {
        **labels_map,
        "week_number": "Numéro de semaine",
        f"{entity_type}s_share": f"% de {entity_label} actifs"
    }
    
    fig_kwargs = {
        "data_frame": df_retention,
        "x": "week_number",
        "y": f"{entity_type}s_share",
        "labels": chart_labels,
        "markers": True,
        "template": "simple_white",
        "title": f"Rétention des {entity_label} sur {num_weeks} semaines <br><sub>% de {entity_label} ayant effectué au moins un trajet par semaine</sub>",
    }
    
    if group_by_col:
        fig_kwargs["color"] = group_by_col
        if color_discrete_map:
            fig_kwargs["color_discrete_map"] = color_discrete_map
    
    fig = px.line(**fig_kwargs)
    
    fig.add_hline(y=100, line_dash="dash", line_color="gray", opacity=0.5)
    
    return fig

In [ ]:
df_retention_specialized = create_retention_analysis(
    df=df_specialized_drivers,
    entity_type="driver",
    num_weeks=12
)

fig_churn_by_campaign_type = plot_retention_analysis(
    df_retention=df_retention_specialized,
    entity_type="driver",
    labels_map=labels_map,
    num_weeks=12
)

fig_churn_by_campaign_type.update_yaxes(showgrid=True)
fig_churn_by_campaign_type.show()

fig_churn_by_campaign_type.write_html(OUTPUT_PATH / "fig_attrition_par_campagne.html")
fig_churn_by_campaign_type.write_image(
    OUTPUT_PATH / "fig_attrition_par_campagne.svg", width=1280, height=720
)

### Distribution des gains


In [ ]:
df_incentives_stats_by_month_driver = (
    df_journeys_raw_with_nearest_stations_full.with_columns(
        (pl.col("incentive_amount") / 100)
        .cum_sum()
        .over(
            partition_by=[
                "driver_identity_key",
                pl.col("start_datetime").dt.truncate("1mo"),
            ],
            order_by="start_datetime",
        )
        .alias("incentive_amount_cumu")
    )
    .group_by([pl.col("start_datetime").dt.truncate("1mo"), "driver_identity_key"])
    .agg(
        pl.col("incentive_amount_cumu").max().alias("incentive_amount_cumu_max"),
        pl.col("is_near_station_fmt").mode().first().alias("is_near_station_fmt"),
    )
)

In [ ]:
breaks = list(range(0, 101, 10))
df_incentives_by_drivers_hist = (
    df_incentives_stats_by_month_driver.group_by(
        [
            pl.col("incentive_amount_cumu_max").cut(breaks=breaks, include_breaks=True),
            "is_near_station_fmt",
        ]
    )
    .agg(pl.col("driver_identity_key").n_unique().alias("driver_count"))
    .with_columns(
        pl.col("incentive_amount_cumu_max").struct.unnest(),
        (
            100
            * pl.col("driver_count")
            / pl.col("driver_count").sum().over("is_near_station_fmt")
        )
        .round(2)
        .alias("share"),
    )
    .sort(["is_near_station_fmt", "breakpoint"])
)

fig_incentives_by_driver_hist = px.bar(
    df_incentives_by_drivers_hist,
    x="category",
    y="share",
    text="share",
    color="is_near_station_fmt",
    color_discrete_map=colors_map,
    category_orders={"is_near_station_fmt": category_order},
    barmode="group",
    labels={**labels_map, "share": "% des conducteurs", "category": "Incitation reçue"},
    template="simple_white",
    title="Distribution des gains mensuels des conducteurs<br><sub>Par tranche de 10€, la première tranche est celle des conducteurs n'ayant perçu aucune incitation.</sub>",
)
# fig_incentives_by_driver_hist.show()
fig_incentives_by_driver_hist.write_html(
    OUTPUT_PATH / "fig_histo_incitation_conducteur.html"
)
fig_incentives_by_driver_hist.write_image(
    OUTPUT_PATH / "fig_histo_incitation_conducteur.svg", width=1280, height=720
)

## Passagers

### Nombre de trajets

In [ ]:
def create_num_passengers_by_num_trips_hist_fig(
    df: pl.DataFrame, step_size: int, max_step: int
) -> go.Figure:
    breaks = range(1, max_step + 1, step_size)

    specialized_passengers = ( # passager faisant 100% de trajet dans une catégorie de distance à une gare
        df.filter(
            pl.col("passenger_first_trip_datetime") >= datetime(2024, 9, 1, tzinfo=ZoneInfo("GMT")),
            pl.col("passenger_first_trip_datetime") <= datetime.now(ZoneInfo("GMT")) - timedelta(days=30),
            pl.col("start_datetime") <= pl.col("passenger_first_trip_datetime") + pl.duration(days=30),
        )
        .group_by("driver_identity_key")
        .agg([
            pl.col("is_near_station_fmt").n_unique().alias("num_categories"),
            pl.col("is_near_station_fmt").first().alias("driver_category")  
        ])
        .filter(pl.col("num_categories") == 1)  
        .select(["driver_identity_key", "driver_category"])
    )

    df = (
        df.filter(
            pl.col("passenger_first_trip_datetime") >= datetime(2024, 9, 1, tzinfo=ZoneInfo("GMT")),
            pl.col("passenger_first_trip_datetime") <= datetime.now(ZoneInfo("GMT")) - timedelta(days=30),
            pl.col("start_datetime") <= pl.col("passenger_first_trip_datetime") + pl.duration(days=30),
        )
        .join(specialized_passengers, on="driver_identity_key", how="inner")  
    )


    station_distance_categories = df.select("is_near_station_fmt").unique().to_series().to_list()

        # Création du DataFrame de toutes les combinaisons possibles
    combinations = pl.DataFrame(
        product(station_distance_categories, breaks), schema=["is_near_station_fmt", "breaks_raw"]
    ).with_columns(
        pl.col("breaks_raw")
        .cut(breaks, include_breaks=True, left_closed=True)
        .struct.unnest()
    )
    data_agg = (
        df.filter(
            pl.col("passenger_first_trip_datetime")
            >= datetime(2024, 9, 1, tzinfo=ZoneInfo("GMT")),
            pl.col("passenger_first_trip_datetime")
            <= datetime.now(ZoneInfo("GMT")) - timedelta(days=30),
            pl.col("start_datetime")
            <= pl.col("passenger_first_trip_datetime") + pl.duration(days=30),
        )
        .group_by(["is_near_station_fmt", "driver_identity_key"])
        .agg(
            pl.len().alias("num_journeys"),
            pl.concat_str(
                pl.col("operator_id"), pl.lit("-"), pl.col("operator_trip_id")
            )
            .n_unique()
            .alias("num_trips"),
        )
        .with_columns(
            pl.col("num_trips").cut(
                breaks=breaks, left_closed=True, include_breaks=True
            )
        )
        .group_by(["is_near_station_fmt", "num_trips"])
        .agg(pl.col("driver_identity_key").n_unique().alias("num_passengers"))
        .with_columns(pl.col("num_trips").struct.unnest())
    )

    data_complete = (
        combinations.join(
            data_agg,
            on=["is_near_station_fmt", "breakpoint"],
            how="left",
        )
        .with_columns(pl.col("num_passengers").fill_null(0))
        .with_columns(
            (
                100
                * pl.col("num_passengers")
                / pl.col("num_passengers").sum().over("is_near_station_fmt")
            )
            .round(2)
            .alias("share_passengers")
        )
        .sort(["breakpoint", "is_near_station_fmt"])
    )


    fig = px.bar(
            data_complete,
            x=data_complete["category"],
            y=data_complete["share_passengers"],
            color="is_near_station_fmt",
            color_discrete_map=colors_map,
            category_orders={"is_near_station_fmt": category_order},
            barmode="group",
            labels=labels_map,
            template="simple_white",
            title="Distribution du nombre de trips effectués sur 30 jours pour chaque catégorie de distance à une gare",
        )


    fig.update_xaxes(title="Nombre de trips")
    fig.update_yaxes(title="% des conducteurs")

    return fig, data_complete


fig_passengers_by_trip_numbers_hist, data_complete = (
    create_num_passengers_by_num_trips_hist_fig(df_journeys_raw_with_nearest_stations_full, step_size=3, max_step=30)
)
fig_passengers_by_trip_numbers_hist.show()

fig_passengers_by_trip_numbers_hist.write_html(
    OUTPUT_PATH / "fig_histo_trajets_passagers_near.html"
)
fig_passengers_by_trip_numbers_hist.write_image(
    OUTPUT_PATH / "fig_histo_trajets_passagers_near.svg", width=1280, height=720
)

### Cohérence comportementale passagers

In [ ]:
fig_passenger_consistency, data_passenger_consistency = create_behavioral_consistency_chart(
    df=df_journeys_raw_with_nearest_stations_full,
    entity_type="passenger",
    category_col="is_near_station",
    category_true_label="proches",
    category_false_label="éloignés",
    labels_map=labels_map,
)

# Avec des breaks personnalisés
fig_custom_breaks, data_custom = create_behavioral_consistency_chart(
    df=df_journeys_raw_with_nearest_stations_full,
    entity_type="driver",
    category_col="is_near_station",
    category_true_label="proches des gares",
    category_false_label="éloignés des gares",
    prop_breaks=[0, 0.2, 0.4, 0.6, 0.8, 1.0],
    labels_map=labels_map,
)

### Rétention

In [ ]:
df_retention_passengers_by_station = create_retention_analysis(
    df=df_journeys_raw_with_nearest_stations_full,
    entity_type="passenger",
    num_weeks=12,
    group_by_col="is_near_station_fmt"
)

fig_churn_by_passenger_near = plot_retention_analysis(
    df_retention=df_retention_passengers_by_station,
    entity_type="passenger",
    labels_map=labels_map,
    num_weeks=12,
    group_by_col="is_near_station_fmt",
    color_discrete_map=colors_map,
)

fig_churn_by_passenger_near.update_yaxes(showgrid=True)
fig_churn_by_passenger_near.show()

fig_churn_by_passenger_near.write_html(OUTPUT_PATH / "fig_churn_by_passenger_near.html")
fig_churn_by_passenger_near.write_image(
    OUTPUT_PATH / "fig_churn_by_passenger_near.svg", width=1280, height=720
)

# Etudes trajets directs et indirects

In [ ]:
df_journeys_near_station = df_journeys_raw_with_nearest_stations_full.filter(
    pl.col("is_near_station") == True
).with_columns(
    (pl.col("line_name_start") == pl.col("line_name_end")).alias("has_direct_train_line")
)
df_journeys_near_station.head()

In [ ]:
df_journeys_near_station_by_week = df_journeys_near_station.group_by(
        [
            pl.col("start_datetime").dt.truncate("1w").alias("week"),
            "has_direct_train_line",
        ]
    ).agg(agg_expressions).sort(pl.col("week"))
df_journeys_near_station_by_month = df_journeys_near_station.group_by(
        [
            pl.col("start_datetime").dt.truncate("1mo").alias("month"),
            "has_direct_train_line",
        ]
    ).agg(agg_expressions).sort(pl.col("month"))



fig_journeys_by_week_direct_vs_undirect = px.line(
    df_journeys_near_station_by_week.sort("has_direct_train_line"),
    x="week",
    y="num_journeys_incentived",
    color="has_direct_train_line",
    color_discrete_map=colors_map,
    template="simple_white",
    labels=labels_map,
    title="Nombre de trajets incités proches d'une gare par semaine",
)
fig_journeys_by_week_direct_vs_undirect.show()

fig_journeys_by_week_direct_vs_undirect
fig_journeys_by_week_direct_vs_undirect.update_yaxes(
    range=[0, df_journeys_near_station_by_week["num_journeys_incentived"].max() - 1.1]
)

fig_journeys_by_week_direct_vs_undirect.write_html(
    OUTPUT_PATH / "fig_journeys_by_week_direct_vs_undirect.html"
)
fig_journeys_by_week_direct_vs_undirect.write_image(
    OUTPUT_PATH / "fig_journeys_by_week_direct_vs_undirect.svg", width=1280, height=720
)

## Distance

In [ ]:
fig_distance_by_month_direct_vs_undirect = px.box(
    df_journeys_near_station.with_columns(
        (pl.col("distance") / 1000).alias("distance_km"), pl.col("start_datetime").dt.strftime("%Y-%m").alias("year_month")
    ).sort("has_direct_train_line"),
    x="year_month",
    y="distance_km",
    color="has_direct_train_line",
    color_discrete_map=colors_map,
    category_orders={"has_direct_train_line": category_order},
    template="simple_white",
    labels=labels_map,
    title="Distribution de la distance par trajet avec incitations par mois - trajets proches de gare",
)
fig_distance_by_month_direct_vs_undirect.update_yaxes(showgrid=True)

fig_distance_by_month_direct_vs_undirect.update_traces(marker=dict(size=0, opacity=0))

fig_distance_by_month_direct_vs_undirect.update_yaxes(

range=[0, 70]

)
fig_distance_by_month_direct_vs_undirect.show()

fig_distance_by_month_direct_vs_undirect

# fig_distance_by_month_direct_vs_undirect.update_yaxes(
#     range=[df_journeys_near_station_by_month["distance_avg"].min() - 1.1, df_journeys_near_station_by_month["distance_avg"].max() + 1.1]
# )

fig_distance_by_month_direct_vs_undirect.write_html(
    OUTPUT_PATH / "fig_distance_by_month_direct_vs_undirect.html"
)
fig_distance_by_month_direct_vs_undirect.write_image(
    OUTPUT_PATH / "fig_distance_by_month_direct_vs_undirect.svg", width=1280, height=720
)

## Montant versé par l'AOM par mois

In [ ]:
fig_amount_aom_by_month_direct_vs_undirect = px.line(
    df_journeys_near_station_by_month.with_columns(
        pl.col("amount_aom_sum").round(1)
    ).sort("has_direct_train_line"),
    x="month",
    y="amount_aom_sum",
   text="amount_aom_sum",
    color="has_direct_train_line",
    color_discrete_map=colors_map,
    category_orders={"has_direct_train_line": category_order},
    template="simple_white",
    labels=labels_map,
    title="Somme des incentives AOM mois",
)
fig_amount_aom_by_month_direct_vs_undirect.show()

fig_amount_aom_by_month_direct_vs_undirect
fig_amount_aom_by_month_direct_vs_undirect.update_traces(textposition="bottom left")

fig_amount_aom_by_month_direct_vs_undirect.update_yaxes(
    range=[df_journeys_near_station_by_month["amount_aom_sum"].min() - 1.1, df_journeys_near_station_by_month["amount_aom_sum"].max() + 1.1]
)

fig_amount_aom_by_month_direct_vs_undirect.write_html(
    OUTPUT_PATH / "fig_amount_aom_by_month_direct_vs_undirect.html"
)
fig_amount_aom_by_month_direct_vs_undirect.write_image(
    OUTPUT_PATH / "fig_amount_aom_by_month_direct_vs_undirect.svg", width=1280, height=720
)

In [ ]:
budget_mensuel = float(BUDGET_IDFM) / float(DUREE_CAMPAGNE_IDFM)

df_journeys_near_station_by_month = df_journeys_near_station_by_month.with_columns([
    (pl.col("amount_aom_sum") / budget_mensuel * 100).alias("pct_budget_mensuel"),
    (pl.col("amount_aom_sum") / budget_mensuel * 100).round(1).alias("pct_budget_mensuel_display")
])

fig_amount_aom_by_month_direct_vs_undirect_as_monthly_budget_pct= px.line(
    df_journeys_near_station_by_month.sort("has_direct_train_line"),
    x="month",
    y="pct_budget_mensuel",  
    text="pct_budget_mensuel_display",  
    color="has_direct_train_line",
    color_discrete_map=colors_map,
    category_orders={"has_direct_train_line": category_order},
    template="simple_white",
    labels=labels_map | {"pct_budget_mensuel": "% du budget mensuel de la campagne d'incitation"},  
    title=f"Coût mensuel des incitations AOM pour les trajets proches de gare en % du budget mensuel de la campagne d'incitation",
)

fig_amount_aom_by_month_direct_vs_undirect_as_monthly_budget_pct.update_traces(
    textposition="bottom left",
    texttemplate="%{text}%"  
)

fig_amount_aom_by_month_direct_vs_undirect_as_monthly_budget_pct.add_hline(
    y=100,
    line_dash="dash",
    line_color="gray",
    line_width=2,
    annotation_text="100% du budget mensuel",
    annotation_position="top right",
    annotation_font_size=11,
    annotation_font_color="gray"
)

fig_amount_aom_by_month_direct_vs_undirect_as_monthly_budget_pct.update_yaxes(
    range=[
        df_journeys_near_station_by_month["pct_budget_mensuel"].min() - 0.5,
        (df_journeys_near_station_by_month["pct_budget_mensuel"].max() + 0.5) 
    ],
    ticksuffix="%"  
)

fig_amount_aom_by_month_direct_vs_undirect_as_monthly_budget_pct.show()

fig_amount_aom_by_month_direct_vs_undirect_as_monthly_budget_pct.write_html(
    OUTPUT_PATH / "fig_amount_aom_by_month_direct_vs_undirect_as_monthly_budget_pct.html"
)

fig_amount_aom_by_month_direct_vs_undirect_as_monthly_budget_pct.write_image(
    OUTPUT_PATH / "fig_amount_aom_by_month_direct_vs_undirect_as_monthly_budget_pct.svg", 
    width=1280, 
    height=720
)

## Attrition

In [ ]:
specialized_drivers_direct_line = filter_specialized_entities(
    df=df_journeys_near_station,
    entity_type="driver",
    category_col="has_direct_train_line",
    period_days=42  
)

df_specialized_drivers_direct = df_journeys_near_station.join(
    specialized_drivers,
    on="driver_identity_key",
    how="inner"
)

df_retention_by_direct_line = create_retention_analysis(
    df=df_specialized_drivers_direct,
    entity_type="driver",
    num_weeks=12,
    group_by_col="has_direct_train_line"
)

fig_churn_has_direct_train_line_vs_undirect = plot_retention_analysis(
    df_retention=df_retention_by_direct_line,
    entity_type="driver",
    labels_map=labels_map,
    num_weeks=12,
    group_by_col="has_direct_train_line",
    color_discrete_map=colors_map,
)

fig_churn_has_direct_train_line_vs_undirect.update_yaxes(showgrid=True)
fig_churn_has_direct_train_line_vs_undirect.show()

fig_churn_has_direct_train_line_vs_undirect.write_html(OUTPUT_PATH / "fig_attrition_tc_direct_ou_non.html")
fig_churn_has_direct_train_line_vs_undirect.write_image(
    OUTPUT_PATH / "fig_attrition_tc_direct_ou_non.svg", width=1280, height=720
)

## Volumes de trajet par ligne pour les trajets en concurence avec TC

### Par semaine

In [ ]:
df_journeys_near_station_with_direct_train_line = df_journeys_near_station.filter(pl.col("has_direct_train_line")).group_by(
        [
            pl.col("start_datetime").dt.truncate("1w").alias("week"),
            "line_name_end"
        ]).agg(agg_expressions).sort(pl.col("week"))
df_journeys_near_station_with_direct_train_line.head()

In [ ]:
fig_journeys_wdirect_line_volume_by_train_line = px.line(
    df_journeys_near_station_with_direct_train_line,
    x="week",
    y="num_journeys_incentived",
    color="line_name_end",
    template="simple_white",
    labels=labels_map,
    title="Nombre de journeys avec incentives en concurence TC par ligne de TC par semaine",
)
fig_journeys_wdirect_line_volume_by_train_line.show()

fig_journeys_wdirect_line_volume_by_train_line
fig_journeys_wdirect_line_volume_by_train_line.update_traces(textposition="top center")

# fig_distance_by_month_direct_vs_undirect.update_yaxes(
#     range=[df_journeys_near_station_by_month["distance_avg"].min() - 1.1, df_journeys_near_station_by_month["distance_avg"].max() + 1.1]
# )

fig_journeys_wdirect_line_volume_by_train_line.write_html(
    OUTPUT_PATH / "fig_journeys_wdirect_line_volume_by_train_line.html"
)
fig_journeys_wdirect_line_volume_by_train_line.write_image(
    OUTPUT_PATH / "fig_journeys_wdirect_line_volume_by_train_line.svg", width=1280, height=720
)

In [ ]:
fig_journeys_wdirect_line_volume_by_train_line_rer = px.line(
    df_journeys_near_station_with_direct_train_line.filter(pl.col("line_name_end").str.contains("RER")),
    x="week",
    y="num_journeys_incentived",
    color="line_name_end",
    template="simple_white",
    labels=labels_map,
    title="Nombre de journeys avec incentives en concurence TC par ligne de TC par semaine",
)
fig_journeys_wdirect_line_volume_by_train_line_rer.show()

fig_journeys_wdirect_line_volume_by_train_line_rer
fig_journeys_wdirect_line_volume_by_train_line_rer.update_traces(textposition="top center")

# fig_distance_by_month_direct_vs_undirect.update_yaxes(
#     range=[df_journeys_near_station_by_month["distance_avg"].min() - 1.1, df_journeys_near_station_by_month["distance_avg"].max() + 1.1]
# )

fig_journeys_wdirect_line_volume_by_train_line_rer.write_html(
    OUTPUT_PATH / "fig_journeys_wdirect_line_volume_by_train_line_rer.html"
)
fig_journeys_wdirect_line_volume_by_train_line_rer.write_image(
    OUTPUT_PATH / "fig_journeys_wdirect_line_volume_by_train_line_rer.svg", width=1280, height=720
)

### Par jour

In [ ]:
df_journeys_near_station_with_direct_train_line_per_day = df_journeys_near_station.filter(pl.col("has_direct_train_line")).group_by(
        [
            pl.col("start_datetime").dt.truncate("1d").alias("day"),
            "line_name_end",
        ]).agg(agg_expressions).sort(pl.col("day"))
df_journeys_near_station_with_direct_train_line_per_day.head()

In [ ]:
fig_journeys_wdirect_line_volume_by_train_line_per_day = px.line(
    df_journeys_near_station_with_direct_train_line_per_day,
    x="day",
    y="num_journeys_incentived",
    color="line_name_end",
    template="simple_white",
    labels=labels_map,
    title="Nombre de journeys avec incentives en concurence TC par ligne de TC par semaine",
)
fig_journeys_wdirect_line_volume_by_train_line_per_day.show()

fig_journeys_wdirect_line_volume_by_train_line_per_day
fig_journeys_wdirect_line_volume_by_train_line_per_day.update_traces(textposition="top center")

# fig_distance_by_month_direct_vs_undirect.update_yaxes(
#     range=[df_journeys_near_station_by_month["distance_avg"].min() - 1.1, df_journeys_near_station_by_month["distance_avg"].max() + 1.1]
# )

fig_journeys_wdirect_line_volume_by_train_line_per_day.write_html(
    OUTPUT_PATH / "fig_journeys_wdirect_line_volume_by_train_line_per_day.html"
)
fig_journeys_wdirect_line_volume_by_train_line_per_day.write_image(
    OUTPUT_PATH / "fig_journeys_wdirect_line_volume_by_train_line_per_day.svg", width=1280, height=720
)

In [ ]:
rer_colors = {
    'RER A': '#E2231A',  
    'RER B': '#5291CE',  
    'RER C': '#F99D1D', 
    'RER D': "#058B42", 
    'RER E': '#C760AD'  
}

fig_journeys_wdirect_line_volume_by_train_line_rer_per_day = px.line(
    df_journeys_near_station_with_direct_train_line_per_day.filter(pl.col("line_name_end").str.contains("RER [ABCD]"),
                                                                   pl.col("day") >=pl.lit("2025-03-01").str.to_date()).sort("line_name_end"),
    x="day",
    y="num_journeys_incentived",
    color="line_name_end",
    color_discrete_map=rer_colors,
    template="simple_white",
    labels=labels_map,
    title="Nombre de trajets avec incentives en concurence TC par ligne de TC par jour",
)
fig_journeys_wdirect_line_volume_by_train_line_rer_per_day.show()

fig_journeys_wdirect_line_volume_by_train_line_rer_per_day
fig_journeys_wdirect_line_volume_by_train_line_rer_per_day.update_traces(textposition="top center")

# fig_distance_by_month_direct_vs_undirect.update_yaxes(
#     range=[df_journeys_near_station_by_month["distance_avg"].min() - 1.1, df_journeys_near_station_by_month["distance_avg"].max() + 1.1]
# )

fig_journeys_wdirect_line_volume_by_train_line_rer_per_day.write_html(
    OUTPUT_PATH / "fig_journeys_wdirect_line_volume_by_train_line_rer_per_day.html"
)
fig_journeys_wdirect_line_volume_by_train_line_rer_per_day.write_image(
    OUTPUT_PATH / "fig_journeys_wdirect_line_volume_by_train_line_rer_per_day.svg", width=1280, height=720
)

### Couple gare départ - gare arrivée les plus fréquents

In [ ]:
df_journeys_near_station_with_direct_train_line_by_start_end = df_journeys_near_station.filter(pl.col("has_direct_train_line")).group_by(
        [
            "nom_gares_start", "nom_gares_end"
        ]).agg(agg_expressions).sort("num_journeys", descending=True)
df_journeys_near_station_with_direct_train_line_by_start_end.head()

In [ ]:
df_top_nearest_stations_w_direct_train_line_couples = (
    df_journeys_near_station_with_direct_train_line_by_start_end
    .with_columns(
        (pl.col("num_journeys") / pl.col("num_journeys").sum()).alias("share_journeys"),
        (pl.col("nom_gares_start") + "_" + pl.col("nom_gares_end")).alias("couple_gares")
    )
    .with_columns(
        pl.format(
            "{}%",
            (100 * pl.col("share_journeys")).round(2)
        ).alias("share_journeys_fmt")
    )
    .sort("num_journeys", descending=True)
)

In [ ]:
fig_journeys_wdirect_line_volume_by_station_start_and_stop = px.bar(
    df_top_nearest_stations_w_direct_train_line_couples.head(10),
    x="couple_gares",
    y="num_journeys",
    text="share_journeys_fmt",
    template="simple_white",
    labels=labels_map,
    title="TOP 10 des trajets entre deux gare les plus fréquents"
    "<br><sub>Dans les barres sont affichées la proportion au regard du total des trajets et la distance moyenne au départ.</sub>",
    height=500,
)

fig_journeys_wdirect_line_volume_by_station_start_and_stop.show()
fig_journeys_wdirect_line_volume_by_station_start_and_stop.write_html(
    OUTPUT_PATH / "fig_journeys_wdirect_line_volume_by_station_start_and_stop.html"
)
fig_journeys_wdirect_line_volume_by_station_start_and_stop.write_image(
    OUTPUT_PATH / "fig_journeys_wdirect_line_volume_by_station_start_and_stop.svg", width=1280, height=720
)

In [ ]:
df_with_cumul = df_top_nearest_stations_w_direct_train_line_couples.with_columns(
    pl.col("share_journeys").cum_sum().alias("cumul_share")
).with_columns(
    (pl.col("cumul_share") * 100).round(2).alias("cumul_percentage")
)

print("Pourcentage cumulé :")
print(f"Top 10  : {df_with_cumul.row(9)[df_with_cumul.columns.index('cumul_percentage')]}%")
print(f"Top 50  : {df_with_cumul.row(49)[df_with_cumul.columns.index('cumul_percentage')]}%") 
print(f"Top 100 : {df_with_cumul.row(99)[df_with_cumul.columns.index('cumul_percentage')]}%")
print(f"Top 500 : {df_with_cumul.row(499)[df_with_cumul.columns.index('cumul_percentage')]}%")
print(f"Top 1000 : {df_with_cumul.row(999)[df_with_cumul.columns.index('cumul_percentage')]}%")

### Carto

In [ ]:
def get_all_trips_with_time_slot_day_night(df_journeys_raw: pl.DataFrame):
    return (
        df_journeys_raw
        .with_columns([
            pl.col("start_datetime").dt.hour().alias("hour"),
            pl.when(pl.col("start_datetime").dt.hour().is_between(5, 21))
            .then(pl.lit("Jour"))
            .otherwise(pl.lit("Nuit"))
            .alias("time_slot")
        ])
    )


In [ ]:
def create_direct_train_percentage_maps(df_journeys_raw: pl.DataFrame):
    """Crée une carte du pourcentage de trains directs par zone H3 et tranche horaire"""
    
    # Ajouter les tranches horaires
    data = get_all_trips_with_time_slot_day_night(df_journeys_raw)
    
    # Calculer le % par cellule H3 et tranche horaire
    df_h3_stats = (
        data
        .with_columns([
            plh3.latlng_to_cell(
                pl.col("start_latitude"), 
                pl.col("start_longitude"), 
                resolution=7
            ).alias("h3_cell")
        ])
        .group_by(["h3_cell", "time_slot"])
        .agg([
            pl.col("has_direct_train_line").sum().alias("with_direct"),
            pl.len().alias("total_journeys")
        ])
        .with_columns([
            (pl.col("with_direct") * 100.0 / pl.col("total_journeys")).alias("direct_percentage"),
            plh3.cell_to_boundary(pl.col("h3_cell")).alias("cell_geom")
        ])
        .with_columns([
            pl.col("cell_geom").map_elements(
                lambda x: shapely.Polygon([[e[1], e[0]] for e in x]),
                return_dtype=pl.Object
            ).alias("cell_geom"),
            pl.col("h3_cell").cast(pl.String)
        ])
    )
    
    # Convertir en GeoDataFrame
    gdf = gpd.GeoDataFrame(
        df_h3_stats.to_pandas()
    ).set_geometry("cell_geom", crs=4326)
    
    # Créer une carte par tranche horaire
    time_slots = sorted(df_h3_stats["time_slot"].unique().to_list())
    figs = {}
    
    for slot in time_slots:
        gdf_slot = gdf[gdf["time_slot"] == slot]
        
        if len(gdf_slot) == 0:
            continue
        
        center = gdf_slot.geometry.unary_union.centroid.coords[0][::-1]
        
        # Colormap verte pour les trains directs
        color_scale = bcm.LinearColormap(
            colors=['#8B0000', '#FF4500', '#FFA500', '#FFFF00', '#90EE90', '#228B22'],
            vmin=0,
            vmax=100,
        )
        color_scale.caption = f"% de trajets avec train direct - {slot}"
        
        m = folium.Map(location=center, zoom_start=9, tiles="openstreetmap")
        
        # Ajouter le titre
        title_html = f'''
        <h3 align="center" style="font-size:20px">
            <b>Pourcentage de trains directs - {slot}</b>
        </h3>
        '''
        m.get_root().html.add_child(folium.Element(title_html))
        
        folium.GeoJson(
            gdf_slot.to_json(),
            style_function=lambda feature: {
                "fillColor": color_scale(feature["properties"]["direct_percentage"]),
                "color": "black",
                "weight": 1,
                "fillOpacity": 0.7,
            },
            tooltip=folium.GeoJsonTooltip(
                fields=["h3_cell", "direct_percentage", "with_direct", "total_journeys"],
                aliases=["Zone H3", "% Direct", "Nb avec direct", "Total trajets"],
                localize=True,
            ),
        ).add_to(m)
        
        color_scale.add_to(m)
        
        figs[slot] = m
    
    return figs

# Exécution
direct_train_maps = create_direct_train_percentage_maps(df_journeys_near_station)

# Sauvegarder les cartes
for slot, fig in direct_train_maps.items():
    filename = slot.replace(" ", "_").replace("(", "").replace(")", "").replace("️", "")
    fig.save(OUTPUT_PATH / f"direct_train_percentage_h3_{filename}.html")
    print(f"Carte sauvegardée : direct_train_percentage_h3_{filename}.html")